# WearECG + ECG-FM Integration Notebook

This notebook is the long-form engineering and scientific scaffold for the current
**WearECG-FM** line of work.

## Table of Contents
1. [Environment Setup](#Environment-Setup)
2. [Problem Definition & Contract](#1.-Problem-Definition-&-Contract)
3. [Experimental Comparison Protocol](#2.-Experimental-Comparison-Protocol)
4. [Core Research Artifacts (Exact Control)](#3.-Core-Research-Artifacts-(Exact-Control))
5. [Source Author Diagnostics & Visualizations](#5.-Source-Author-Diagnostics-&-Visualizations)
6. [Sunnybrook Metadata Extraction](#6.-Sunnybrook-Metadata-Extraction)
7. [Data Pipeline & Loading](#7.-Data-Pipeline-&-Loading)
8. [Model Architectures](#8.-Model-Architectures)
9. [Sunnybrook Clinical Evaluation (Baseline WearECG)](#9.-Sunnybrook-Clinical-Evaluation-(Baseline-WearECG))
10. [Main Execution Context](#10.-Main-Execution-Context)
11. [Full Source Appendix](#11.-Full-Source-Appendix)


## Environment Setup

Injecting project roots into `sys.path` to ensure all imports ('src', 'utils', etc.) function correctly.


In [1]:
import sys
import os
from pathlib import Path

project_root = '/home/mithunmanivannan'
repo_root = '/home/mithunmanivannan/third_party/WearECG-reconstruction'

for p in [project_root, repo_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import warnings
print(f"Environment initialized. PyTorch version: {torch.__version__}")


Environment initialized. PyTorch version: 2.6.0+cu124


## 1. Problem Definition & Contract


## 2. Experimental Comparison Protocol


## 3. Core Research Artifacts (Exact Control)


In [2]:
import json
CONTROL_SUMMARY_PATH = Path("/home/mithunmanivannan/checkpoints/ul_ecg/engineering_wearecg_exact_II-V1-V5_bs80_lf1.5/best_summary.json")
if CONTROL_SUMMARY_PATH.exists():
    summary = json.loads(CONTROL_SUMMARY_PATH.read_text())
    print("### Exact WearECG Control Metrics")
    m = summary.get('metrics', summary.get('best_metrics', {}))
    display(pd.DataFrame([m]).T)


### Exact WearECG Control Metrics


,0
val/align_loss,0.000000
val/clinical_reg_full_chest_rwave_progression_corr,0.908155
val/clinical_reg_full_chest_rwave_progression_mae,0.146652
val/clinical_reg_recon_V4_mean_beat_corr,0.977384
val/clinical_reg_recon_V4_mean_beat_rmse,0.067910
...,...
val/rmse_reg_aVR,0.053622
val/rmse_reg_chest_mean,0.090831
val/rmse_reg_lateral_mean,0.090942
val/stft_loss,0.000000


## 5. Source Author Diagnostics & Visualizations


In [3]:
def plot_ecg_comparison_v2(ori_data, samples, index, title="ECG Comparison"):
    standard_leads = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
    fig, axes = plt.subplots(nrows=12, ncols=1, figsize=(10, 15), constrained_layout=True)
    for i in range(12):
        axes[i].plot(ori_data[index, :, i], color="blue", alpha=0.5, label="GT")
        axes[i].plot(samples[index, :, i], color="red", alpha=0.7, label="Pred")
    plt.show()
print("Diagnostics initialized.")


Diagnostics initialized.


## 6. Sunnybrook Metadata Extraction

This section defines the extraction logic for Philips ECG XML files.


In [4]:
"""Sunnybrook ECG metadata extraction pipeline.

This module parses Philips ECG XML files, enforces guardrails around data
integrity and privacy, and exports sanitized feature vectors for downstream
conditioning in the reconstruction model.

Guardrails referenced:
- G1.1: XML validation and error logging
- G1.2: Removal of protected health information
- G1.3: Explicit missingness indicators (no silent imputation)
- G1.4: Physiologic range checks (flag outliers, keep values)
- G1.5: Final CSV must be readable with strict schema (no NaNs written)
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import logging
from logging.handlers import RotatingFileHandler
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple, Union
import xml.etree.ElementTree as ET

import pandas as pd

try:
    import yaml
except ImportError:  # pragma: no cover - yaml is optional
    yaml = None


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PHI_FIELDS = {
    "patient_name",
    "name",
    "mrn",
    "medical_record_number",
    "provider_name",
    "email",
    "phone",
    "dob",
}

DEFAULT_OUTLIER_THRESHOLDS: Dict[str, Tuple[Optional[float], Optional[float]]] = {
    "age_years": (0, 120),
    "qrs_duration_ms": (50, 300),
    "qt_interval_ms": (200, 600),
    "pr_interval_ms": (80, 400),
    "qrs_axis_deg": (-180, 180),
    "t_wave_axis_deg": (-180, 180),
    "sampling_rate_hz": (100, 2000),
    "heart_rate_bpm": (20, 200),
}

DEFAULT_XPATH_MAP: Dict[str, List[str]] = {
    # Demographics
    "patient_id": [
        ".//{ns}PatientID",
        ".//{ns}patientId",
        ".//patient/id",
        ".//Case/CaseId",
    ],
    "encounter_id": [
        ".//{ns}VisitID",
        ".//visitId",
        ".//Case/EncounterId",
    ],
    "age_years": [
        ".//{ns}PatientAge",
        ".//patientAge",
        ".//Patient/Age",
    ],
    "sex_code": [
        ".//{ns}PatientSex",
        ".//patientSex",
        ".//Patient/Sex",
    ],
    "pacemaker_status": [
        ".//{ns}Pacemaker",
        ".//pacemaker",
        ".//Findings/Pacemaker",
    ],
    # Acquisition
    "sampling_rate_hz": [
        ".//{ns}SampleBase",
        ".//SamplingRate",
        ".//acquisition/SamplingRate",
    ],
    "lowpass_hz": [
        ".//{ns}LowPassFilter",
        ".//Filters/LowPass",
    ],
    "hipass_hz": [
        ".//{ns}HighPassFilter",
        ".//Filters/HighPass",
    ],
    "machine_model": [
        ".//{ns}MachineModel",
        ".//Device/Model",
    ],
    # Intervals
    "qrs_duration_ms": [
        ".//{ns}QRSDuration",
        ".//Intervals/QRS",
    ],
    "qt_interval_ms": [
        ".//{ns}QTInterval",
        ".//Intervals/QT",
    ],
    "pr_interval_ms": [
        ".//{ns}PRInterval",
        ".//Intervals/PR",
    ],
    # Axes
    "qrs_axis_deg": [
        ".//{ns}QRSAxis",
        ".//Axis/QRS",
    ],
    "t_wave_axis_deg": [
        ".//{ns}TAxis",
        ".//Axis/T",
    ],
}

DEFAULT_NULL_TOKEN = "NULL"


@dataclass
class SunnybrookExtractorConfig:
    """Configuration container for the Sunnybrook feature extractor."""

    input_dir: Path
    features_csv: Path
    raw_csv: Path
    error_log: Path
    warning_csv: Path
    qa_report_path: Path
    qa_figure_path: Path
    hash_salt: str = "sunnybrook-default-salt"
    null_token: str = DEFAULT_NULL_TOKEN
    outlier_thresholds: Dict[str, Tuple[Optional[float], Optional[float]]] = field(
        default_factory=lambda: DEFAULT_OUTLIER_THRESHOLDS.copy()
    )
    xpath_map: Dict[str, List[str]] = field(
        default_factory=lambda: {k: v[:] for k, v in DEFAULT_XPATH_MAP.items()}
    )
    ensure_directories: bool = True

    @classmethod
    def from_mapping(cls, data: Dict[str, Union[str, Dict, List]]) -> "SunnybrookExtractorConfig":
        """Build config from dictionary, coercing paths to Path objects."""

        def to_path(value: Union[str, Path]) -> Path:
            return Path(value).expanduser().resolve()

        required_keys = [
            "input_dir",
            "features_csv",
            "raw_csv",
            "error_log",
            "warning_csv",
            "qa_report_path",
            "qa_figure_path",
        ]

        missing = [k for k in required_keys if k not in data]
        if missing:
            raise ValueError(f"Missing required config keys: {missing}")

        mapped = {
            "input_dir": to_path(data["input_dir"]),
            "features_csv": to_path(data["features_csv"]),
            "raw_csv": to_path(data["raw_csv"]),
            "error_log": to_path(data["error_log"]),
            "warning_csv": to_path(data["warning_csv"]),
            "qa_report_path": to_path(data["qa_report_path"]),
            "qa_figure_path": to_path(data["qa_figure_path"]),
        }
        optional_keys = {"hash_salt", "null_token", "outlier_thresholds", "xpath_map", "ensure_directories"}
        for key in optional_keys:
            if key in data:
                mapped[key] = data[key]

        return cls(**mapped)

    @classmethod
    def from_file(cls, path: Union[str, Path]) -> "SunnybrookExtractorConfig":
        """Load configuration from JSON or YAML file."""
        path = Path(path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"Config file not found: {path}")

        if path.suffix.lower() in {".yaml", ".yml"}:
            if yaml is None:
                raise RuntimeError("pyyaml not available but YAML config requested")
            with path.open("r", encoding="utf-8") as fh:
                data = yaml.safe_load(fh)
        else:
            with path.open("r", encoding="utf-8") as fh:
                data = json.load(fh)

        if not isinstance(data, dict):
            raise ValueError(f"Config file must contain mapping, got {type(data)}")

        return cls.from_mapping(data)


# ---------------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------------

def _build_logger(name: str, error_log: Path) -> logging.Logger:
    """Create a logger that writes INFO to stdout and errors to file."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)

    if logger.handlers:
        return logger  # reuse existing logger

    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    formatter = logging.Formatter("[%(asctime)s] %(levelname)s %(name)s: %(message)s")
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    error_log.parent.mkdir(parents=True, exist_ok=True)
    file_handler = RotatingFileHandler(error_log, maxBytes=5 * 1024 * 1024, backupCount=3)
    file_handler.setLevel(logging.ERROR)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    logger.propagate = False
    return logger


def _hash_value(raw: str, salt: str) -> str:
    """Return SHA256 hash of the supplied value using provided salt."""
    hasher = hashlib.sha256()
    hasher.update(salt.encode("utf-8"))
    hasher.update(str(raw).encode("utf-8"))
    return hasher.hexdigest()


def _md5_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Compute md5 hash of a file (used for provenance)."""
    digest = hashlib.md5()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _ensure_warning_csv(path: Path) -> None:
    """Ensure warning CSV exists with header."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="", encoding="utf-8") as fh:
            writer = csv.writer(fh)
            writer.writerow(["timestamp_utc", "file", "field", "value", "expected_range", "note"])


# ---------------------------------------------------------------------------
# Extractor implementation
# ---------------------------------------------------------------------------

class SunnybrookFeatureExtractor:
    """Parser and processor for Sunnybrook Philips ECG XML metadata."""

    def __init__(self, config: SunnybrookExtractorConfig) -> None:
        self.config = config
        self.logger = _build_logger(self.__class__.__name__, config.error_log)
        self.invalid_files: List[Path] = []
        self.stats: Dict[str, int] = {
            "total_files": 0,
            "parsed": 0,
            "skipped": 0,
            "outliers": 0,
        }
        self.xpath_map = config.xpath_map
        self.null_token = config.null_token
        self.hash_salt = config.hash_salt
        _ensure_warning_csv(config.warning_csv)
        if config.ensure_directories:
            config.features_csv.parent.mkdir(parents=True, exist_ok=True)
            config.raw_csv.parent.mkdir(parents=True, exist_ok=True)
            config.qa_report_path.parent.mkdir(parents=True, exist_ok=True)
            config.qa_figure_path.parent.mkdir(parents=True, exist_ok=True)

    # ---------------------------- XML helpers ----------------------------

    def validate_schema(self, xml_path: Path) -> Tuple[bool, Optional[ET.ElementTree], Dict[str, str]]:
        """Validate XML structure (G1.1). Returns (is_valid, tree, namespaces)."""
        try:
            tree = ET.parse(xml_path)
        except (ET.ParseError, OSError) as exc:
            self.logger.error("Failed to parse %s: %s", xml_path, exc)
            self.invalid_files.append(xml_path)
            self.stats["skipped"] += 1
            return False, None, {}

        root = tree.getroot()
        ns_map = self._build_namespace_map(root)
        return True, tree, ns_map

    @staticmethod
    def _build_namespace_map(root: ET.Element) -> Dict[str, str]:
        """Extract namespace mappings from the XML root element."""
        ns_map: Dict[str, str] = {}
        if root.tag.startswith("{"):
            uri = root.tag.split("}")[0][1:]
            ns_map["ns"] = uri
        # Additional namespace definitions on elements
        for key, value in root.attrib.items():
            if key.startswith("xmlns:"):
                prefix = key.split("xmlns:")[1]
                ns_map[prefix] = value
            elif key == "xmlns":
                ns_map["ns"] = value
        if "ns" not in ns_map:
            ns_map["ns"] = ""  # fallback for non-namespaced docs
        return ns_map

    def _find_text(self, root: ET.Element, ns_map: Dict[str, str], candidates: Iterable[str]) -> Optional[str]:
        """Attempt multiple xpaths (with namespace substitution) to retrieve text."""
        for raw_xpath in candidates:
            xpath = (
                raw_xpath.format(ns="{%s}" % ns_map["ns"])
                if "{ns}" in raw_xpath and ns_map.get("ns")
                else raw_xpath.replace("{ns}", "")
            )
            element = root.find(xpath, namespaces=ns_map if ns_map.get("ns") else None)
            if element is None:
                continue
            text = element.text
            if text is not None:
                stripped = text.strip()
                if stripped:
                    return stripped
        return None

    def parse_single_ecg(
        self, tree: ET.ElementTree, xml_path: Path, ns_map: Dict[str, str]
    ) -> Optional[Dict[str, Union[str, float, int]]]:
        """Extract metadata from a single ECG XML file (G1.2-G1.4)."""
        root = tree.getroot()
        record: Dict[str, Union[str, float, int]] = {
            "file": str(xml_path.resolve()),
            "processing_timestamp": datetime.now(timezone.utc).isoformat(),
            "source_md5": _md5_file(xml_path),
        }

        missing_flags: Dict[str, int] = {}

        # Extract fields defined in xpath_map
        for field, candidates in self.xpath_map.items():
            value = self._find_text(root, ns_map, candidates)
            if value is None:
                record[field] = self.null_token
                missing_flags[f"is_missing_{field}"] = 1
                continue
            cleaned = value.strip()
            record[field] = cleaned
            missing_flags[f"is_missing_{field}"] = 0

        # Hash identifiers (prevent PHI leakage)
        for key in ["patient_id", "encounter_id", "machine_model"]:
            if key in record:
                value = record[key]
                if value == self.null_token:
                    continue
                hashed = _hash_value(value, self.hash_salt)
                record[f"{key}_hash"] = hashed
                del record[key]
                missing_flags.setdefault(f"is_missing_{key}", 0)

        # Ensure encounter hash fallback
        if "encounter_id_hash" not in record:
            fallback = _hash_value(xml_path.stem, self.hash_salt)
            record["encounter_id_hash"] = fallback
            missing_flags.setdefault("is_missing_encounter_id", 1)

        # Pacemaker categorical normalization (0=unknown,1=no,2=yes)
        record["pacemaker_status"] = self._normalize_categorical(record.get("pacemaker_status", self.null_token))
        missing_flags.setdefault("is_missing_pacemaker_status", 0 if record["pacemaker_status"] != 0 else 1)

        # Sex code normalization
        record["sex_code"] = self._normalize_sex(record.get("sex_code", self.null_token))
        missing_flags.setdefault("is_missing_sex_code", 0 if record["sex_code"] != 0 else 1)

        # Numeric conversions + outlier logging (G1.4)
        numeric_fields = [
            "age_years",
            "sampling_rate_hz",
            "lowpass_hz",
            "hipass_hz",
            "qrs_duration_ms",
            "qt_interval_ms",
            "pr_interval_ms",
            "qrs_axis_deg",
            "t_wave_axis_deg",
            "heart_rate_bpm",
        ]
        for field in numeric_fields:
            raw_value = record.get(field, self.null_token)
            if raw_value == self.null_token:
                continue
            converted = self._safe_float(raw_value)
            if converted is None:
                self.logger.warning("Non-numeric value for %s in %s: %s", field, xml_path, raw_value)
                record[field] = self.null_token
                missing_flags[f"is_missing_{field}"] = 1
                continue
            record[field] = converted
            bounds = self.config.outlier_thresholds.get(field)
            if bounds is not None:
                low, high = bounds
                out_of_range = (low is not None and converted < low) or (high is not None and converted > high)
                if out_of_range:
                    self._log_outlier(xml_path, field, converted, bounds, note="Physiologic range violation")
                    self.stats["outliers"] += 1

        # Check PHI
        self._assert_no_phi(record)

        # Merge missing indicators
        record.update(missing_flags)

        namespace = ns_map.get("ns", "")
        prefix = f"{{{namespace}}}" if namespace else ""

        # Enrich demographics
        patient_info = root.find(f".//{prefix}patient/{prefix}generalpatientdata")
        if patient_info is not None:
            years_el = patient_info.find(f".//{prefix}age/{prefix}years")
            if years_el is not None and years_el.text:
                try:
                    record["age_years"] = float(years_el.text)
                    missing_flags[f"is_missing_age_years"] = 0
                except ValueError:
                    pass
            sex_el = patient_info.find(f".//{prefix}sex")
            if sex_el is not None and sex_el.text:
                sex_text = sex_el.text.strip().lower()
                if "male" in sex_text:
                    record["sex_code"] = 1
                elif "female" in sex_text:
                    record["sex_code"] = 2
                elif "unknown" in sex_text:
                    record["sex_code"] = 0
                missing_flags["is_missing_sex_code"] = 0

        # Activity measurements
        measurements = root.find(f".//{prefix}internalmeasurements/{prefix}crossleadmeasurements")
        measurement_map = {
            "meanventrate": "heart_rate_bpm",
            "meanqrsdur": "qrs_duration_ms",
            "meanqtint": "qt_interval_ms",
            "meanprint": "pr_interval_ms",
            "qrsfrontaxis": "qrs_axis_deg",
            "tfrontaxis": "t_wave_axis_deg",
        }
        if measurements is not None:
            for xml_tag, key in measurement_map.items():
                el = measurements.find(f"{prefix}{xml_tag}")
                if el is not None and el.text:
                    try:
                        record[key] = float(el.text)
                        missing_flags[f"is_missing_{key}"] = 0
                    except ValueError:
                        continue
        if "heart_rate_bpm" not in record:
            record["heart_rate_bpm"] = self.null_token
        missing_flags.setdefault("is_missing_heart_rate_bpm", 1 if record["heart_rate_bpm"] == self.null_token else 0)

        # Acquisition metadata
        wave = root.find(f".//{prefix}waveforms/{prefix}parsedwaveforms")
        if wave is not None:
            attr_map = {
                "samplespersecond": "sampling_rate_hz",
                "hipass": "hipass_hz",
                "lowpass": "lowpass_hz",
            }
            for attr, key in attr_map.items():
                value = wave.get(attr)
                if value not in (None, "", self.null_token):
                    try:
                        record[key] = float(value)
                        missing_flags[f"is_missing_{key}"] = 0
                    except ValueError:
                        continue

        return record

    @staticmethod
    def _safe_float(value: Union[str, float, int]) -> Optional[float]:
        """Convert value to float, returning None on failure."""
        try:
            return float(str(value).strip())
        except (TypeError, ValueError):
            return None

    def _normalize_categorical(self, raw: Union[str, int]) -> int:
        """Normalize pacemaker categorical values."""
        if isinstance(raw, int):
            if raw in {0, 1, 2}:
                return raw
            return 0
        if not isinstance(raw, str):
            return 0
        lowered = raw.strip().lower()
        if lowered in {"1", "no", "false", "absent"}:
            return 1
        if lowered in {"2", "yes", "true", "present"}:
            return 2
        if lowered in {"0", "unknown", "", "n/a"}:
            return 0
        return 0

    def _normalize_sex(self, raw: Union[str, int]) -> int:
        """Normalize sex categorical values (0=unknown,1=male,2=female)."""
        if isinstance(raw, int):
            if raw in {0, 1, 2}:
                return raw
            return 0
        lowered = str(raw).strip().lower()
        if lowered in {"m", "male", "1"}:
            return 1
        if lowered in {"f", "female", "2"}:
            return 2
        return 0

    def _log_outlier(
        self,
        xml_path: Path,
        field: str,
        value: Union[int, float],
        bounds: Tuple[Optional[float], Optional[float]],
        note: str,
    ) -> None:
        """Append outlier information to CSV (G1.4)."""
        with self.config.warning_csv.open("a", newline="", encoding="utf-8") as fh:
            writer = csv.writer(fh)
            writer.writerow(
                [
                    datetime.now(timezone.utc).isoformat(),
                    str(xml_path.resolve()),
                    field,
                    value,
                    str(bounds),
                    note,
                ]
            )

    def _assert_no_phi(self, record: Dict[str, Union[str, float, int]]) -> None:
        """Ensure no PHI-like keys are present (G1.2)."""
        present = PHI_FIELDS.intersection(record.keys())
        if present:
            raise ValueError(f"PHI fields detected in record: {present}")

    # ---------------------------- orchestration ----------------------------

    def process_all(self, sample_n: Optional[int] = None) -> pd.DataFrame:
        """Process all XML files and return sanitized DataFrame."""
        xml_files = sorted(Path(self.config.input_dir).glob("*.xml"))
        if not xml_files:
            self.logger.warning("No XML files found under %s", self.config.input_dir)
            empty_df = pd.DataFrame()
            return empty_df, empty_df

        records: List[Dict[str, Union[str, float, int]]] = []
        for idx, xml_path in enumerate(xml_files, start=1):
            if sample_n is not None and idx > sample_n:
                break
            self.stats["total_files"] += 1
            if idx % 25 == 0:
                self.logger.info("Processing file %d of %d", idx, len(xml_files))
            valid, tree, ns_map = self.validate_schema(xml_path)
            if not valid or tree is None:
                continue
            record = self.parse_single_ecg(tree, xml_path, ns_map)
            if record is None:
                self.stats["skipped"] += 1
                continue
            records.append(record)
            self.stats["parsed"] += 1

        raw_df, export_df = self._build_dataframe(records)
        self._write_outputs(raw_df, export_df)
        return raw_df, export_df

    def _build_dataframe(self, records: List[Dict[str, Union[str, float, int]]]) -> pd.DataFrame:
        """Build DataFrame with consistent column ordering."""
        if not records:
            empty_df = pd.DataFrame()
            return empty_df, empty_df

        df = pd.DataFrame(records)
        df["processing_timestamp"] = pd.to_datetime(df["processing_timestamp"], utc=True)

        # Ensure missing indicator columns exist
        missing_cols = [col for col in df.columns if col.startswith("is_missing_")]
        for field in self.xpath_map.keys():
            flag_col = f"is_missing_{field}"
            if flag_col not in missing_cols:
                df[flag_col] = 1

        # Convert numeric columns to floats, others to strings
        numeric_cols = [
            "age_years",
            "sampling_rate_hz",
            "lowpass_hz",
            "hipass_hz",
            "qrs_duration_ms",
            "qt_interval_ms",
            "pr_interval_ms",
            "qrs_axis_deg",
            "t_wave_axis_deg",
            "heart_rate_bpm",
        ]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # Replace NaN with null token for export
        df_export = df.copy()
        for col in numeric_cols:
            if col in df_export.columns:
                df_export[col] = df_export[col].apply(
                    lambda x: self.null_token if pd.isna(x) else f"{x:.6g}"
                )

        additional_columns = [
            "file",
            "encounter_id_hash",
            "pacemaker_status",
            "sex_code",
            "processing_timestamp",
            "source_md5",
        ]
        for col in additional_columns:
            if col not in df_export.columns:
                df_export[col] = self.null_token

        # Append hashed machine model if present
        if "machine_model_hash" not in df_export.columns:
            df_export["machine_model_hash"] = self.null_token

        # Ensure boolean indicators exported as 0/1 strings
        indicator_cols = [c for c in df_export.columns if c.startswith("is_missing_")]
        for col in indicator_cols:
            df_export[col] = df_export[col].fillna(1).astype(int).astype(str)

        # Convert processing timestamp to ISO string for export
        df_export["processing_timestamp"] = df["processing_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")

        # Ensure column order: base identifiers, numeric, categorical, indicators
        identifier_cols = [
            "file",
            "encounter_id_hash",
            "machine_model_hash",
            "processing_timestamp",
            "source_md5",
        ]
        categorical_cols = ["sex_code", "pacemaker_status"]
        column_order = (
            identifier_cols
            + [col for col in numeric_cols if col in df_export.columns]
            + categorical_cols
            + sorted(indicator_cols)
        )
        # Add any remaining columns (e.g., fallback hashes)
        remaining = [col for col in df_export.columns if col not in column_order]
        column_order.extend(sorted(remaining))
        df_export = df_export[column_order]
        return df, df_export

    def _write_outputs(self, raw_df: pd.DataFrame, export_df: pd.DataFrame) -> None:
        """Persist raw and sanitized DataFrames (G1.5 validation)."""
        if raw_df.empty:
            self.logger.warning("No records parsed; skipping CSV export.")
            return

        export_df = export_df.fillna(self.null_token)

        raw_df.to_csv(self.config.raw_csv, index=False)
        export_df.to_csv(self.config.features_csv, index=False)

        # Validate exported CSV by reading back in
        try:
            validation_df = pd.read_csv(
                self.config.features_csv,
                dtype=str,
                keep_default_na=False,
                na_values=[],
            )
        except Exception as exc:  # pragma: no cover - defensive
            self.logger.error("Failed to read back exported CSV (G1.5 violation): %s", exc)
            raise

        expected_cols = set(export_df.columns)
        if set(validation_df.columns) != expected_cols:
            raise ValueError(
                f"CSV validation failed: columns mismatch. "
                f"expected={sorted(expected_cols)}, found={sorted(validation_df.columns)}"
            )

        if validation_df.isnull().any().any():
            null_cols = validation_df.columns[validation_df.isnull().any()].tolist()
            raise ValueError(
                f"CSV validation failed: detected NaN values after export (G1.5) in columns {null_cols}"
            )

        self.logger.info(
            "Exported raw features to %s and sanitized features to %s",
            self.config.raw_csv,
            self.config.features_csv,
        )

        self._generate_quality_artifacts(raw_df, export_df)

    # ---------------------------- QA / Reporting ----------------------------

    def _generate_quality_artifacts(self, raw_df: pd.DataFrame, export_df: pd.DataFrame) -> None:
        """Generate QA report and distribution plots (G1.3, G1.4)."""
        import matplotlib.pyplot as plt  # Local import to avoid heavy dependency at import time

        report_lines: List[str] = []
        report_lines.append(f"Sunnybrook Metadata QA Report - {datetime.now(timezone.utc).isoformat()}")
        report_lines.append("=" * 72)
        report_lines.append(f"Total samples processed: {len(export_df)}")
        report_lines.append(f"Files skipped (invalid XML): {self.stats['skipped']}")
        report_lines.append(f"Outliers flagged: {self.stats['outliers']}")
        report_lines.append("")

        # Demographics summary
        def summarize(series: pd.Series) -> str:
            if series.dropna().empty:
                return "no data"
            return (
                f"mean={series.mean():.2f}, std={series.std():.2f}, "
                f"min={series.min():.2f}, max={series.max():.2f}, n={series.count()}"
            )

        numeric_cols = [
            "age_years",
            "qrs_duration_ms",
            "qt_interval_ms",
            "pr_interval_ms",
            "heart_rate_bpm",
        ]
        for col in numeric_cols:
            if col in raw_df.columns:
                series = pd.to_numeric(raw_df[col], errors="coerce")
                report_lines.append(f"{col}: {summarize(series)}")
        report_lines.append("")

        # Missingness stats
        indicator_cols = [c for c in export_df.columns if c.startswith("is_missing_")]
        report_lines.append("Missingness (% of records):")
        for col in sorted(indicator_cols):
            percent_missing = export_df[col].astype(float).mean() * 100.0
            report_lines.append(f"  {col}: {percent_missing:.2f}%")
        report_lines.append("")

        # Guardrail compliance summary
        report_lines.append("Guardrail Compliance:")
        report_lines.append("  G1.1 XML validation executed (see logs/parsing_errors.log)")
        report_lines.append("  G1.2 PHI scrubbing enforced via hashed identifiers")
        report_lines.append("  G1.3 Missingness indicators exported")
        report_lines.append("  G1.4 Physiology checks logged to warnings/outliers.csv")
        report_lines.append("  G1.5 CSV round-trip validation passed")
        report_lines.append("")

        self.config.qa_report_path.write_text("\n".join(report_lines), encoding="utf-8")

        # Distribution plots
        plt.style.use("ggplot")
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        plot_fields = ["age_years", "qrs_duration_ms", "qt_interval_ms"]
        thresholds = self.config.outlier_thresholds
        for ax, field in zip(axes, plot_fields):
            series = pd.to_numeric(raw_df.get(field, pd.Series(dtype=float)), errors="coerce").dropna()
            if series.empty:
                ax.text(0.5, 0.5, f"No data for {field}", ha="center", va="center")
                continue
            ax.hist(series, bins=20, color="#2E86AB", alpha=0.8)
            ax.set_title(field)
            ax.set_xlabel(field)
            ax.set_ylabel("Count")
            bounds = thresholds.get(field)
            if bounds:
                low, high = bounds
                if low is not None:
                    ax.axvline(low, color="red", linestyle="--", label="min threshold")
                if high is not None:
                    ax.axvline(high, color="red", linestyle="--", label="max threshold")
            ax.axvline(series.mean(), color="black", linestyle="-", linewidth=1, label="mean")
            ax.legend(loc="upper right")

        fig.tight_layout()
        fig.savefig(self.config.qa_figure_path, dpi=200)
        plt.close(fig)

        self.logger.info("Wrote QA report to %s and distributions to %s", self.config.qa_report_path, self.config.qa_figure_path)

    # ------------------------------------------------------------------
    # CLI
    # ------------------------------------------------------------------

    @staticmethod
    def add_cli(parser: argparse.ArgumentParser) -> None:
        """Register CLI options."""
        parser.add_argument("--config", required=True, help="Path to extractor config (JSON or YAML).")
        parser.add_argument("--sample_n", type=int, default=None, help="Optional cap on number of XML files for smoke test.")

    @classmethod
    def from_cli(cls) -> "SunnybrookFeatureExtractor":
        """Instantiate extractor using CLI arguments."""
        parser = argparse.ArgumentParser(description="Sunnybrook ECG metadata extractor")
        cls.add_cli(parser)
        args = parser.parse_args()
        config = SunnybrookExtractorConfig.from_file(args.config)
        extractor = cls(config)
        extractor._cli_sample_n = args.sample_n  # attach for run()
        return extractor

    def run(self, sample_n: Optional[int] = None) -> None:
        """Execute extraction pipeline and update PHASE_STATUS log."""
        if sample_n is None:
            sample_n = getattr(self, "_cli_sample_n", None)

        raw_df, export_df = self.process_all(sample_n=sample_n)
        if export_df.empty:
            self.logger.warning("No data exported; skipping PHASE_STATUS update.")
            return

        # Update phase status file if available
        status_path = Path("PHASE_STATUS.txt")
        if status_path.exists():
            self._update_phase_status(status_path, len(export_df))

    def _update_phase_status(self, status_path: Path, num_records: int) -> None:
        """Append Phase 1 completion summary to PHASE_STATUS.txt."""
        timestamp = datetime.now(timezone.utc).isoformat()
        summary_lines = [
            "",
            f"[{timestamp}] Phase 1 Extraction Summary",
            f"Processed files: {self.stats['total_files']}",
            f"Parsed records: {self.stats['parsed']}",
            f"Skipped files: {self.stats['skipped']}",
            f"Outliers flagged: {self.stats['outliers']}",
            f"Exported records: {num_records}",
            f"Features CSV: {self.config.features_csv}",
            f"Raw CSV: {self.config.raw_csv}",
            f"Error log: {self.config.error_log}",
            f"Warnings CSV: {self.config.warning_csv}",
            "Guardrail checks:",
            "  [x] G1.1",
            "  [x] G1.2",
            "  [x] G1.3",
            "  [x] G1.4",
            "  [x] G1.5",
        ]
        with status_path.open("a", encoding="utf-8") as fh:
            fh.write("\n".join(summary_lines) + "\n")


def main() -> None:
    extractor = SunnybrookFeatureExtractor.from_cli()
    extractor.run()

## 7. Data Pipeline & Loading


In [5]:
import torch
from utils.io_utils import instantiate_from_config


def build_dataloader_train(config, args=None):  # train dataset / unconditional sample
    batch_size = config["dataloader"]["batch_size"]
    # 从配置中读取 batch_size 和 shuffle
    jud = config["dataloader"]["shuffle"]
    # 设置训练集参数中的 output_dir，为保存路径
    config["dataloader"]["train_dataset"]["params"]["output_dir"] = args.save_dir

    # 将 synthesis_channels 参数加入训练集配置
    if args.mode == "synthesis":
        config["dataloader"]["train_dataset"]["params"][
            "synthesis_channels"
        ] = args.synthesis_channels

    dataset = instantiate_from_config(config["dataloader"]["train_dataset"])

    # 包装数据集，设置批量大小、是否打乱、是否丢弃最后不足一个 batch 的数据
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=jud,
        num_workers=0,
        pin_memory=True,
        sampler=None,
        drop_last=jud,
    )

    dataload_info = {"dataloader": dataloader, "dataset": dataset}

    return dataload_info


def build_dataloader_sample(config, args=None):  # sampling dataset / test dataset
    batch_size = config["dataloader"]["sample_size"]
    config["dataloader"]["test_dataset"]["params"]["output_dir"] = args.save_dir
    # 根据不同的模式（infill、predict、synthesis），设置不同的参数到测试集配置里
    if args.mode == "infill":
        config["dataloader"]["test_dataset"]["params"][
            "missing_ratio"
        ] = args.missing_ratio
    elif args.mode == "predict":
        config["dataloader"]["test_dataset"]["params"]["predict_length"] = args.pred_len
    elif args.mode == "synthesis":
        config["dataloader"]["test_dataset"]["params"][
            "synthesis_channels"
        ] = args.synthesis_channels
    else:
        print(f"Unmatched mode: {args.mode}")

    test_dataset = instantiate_from_config(config["dataloader"]["test_dataset"])

    dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        sampler=None,
        drop_last=False,
    )

    dataload_info = {"dataloader": dataloader, "dataset": test_dataset}

    return dataload_info


if __name__ == "__main__":
    pass


## 8. Model Architectures


### 8A. Baseline WearECG VAE (vae.py)


In [6]:
"""Near-verbatim WearECG VAE modules and loss.

This file intentionally mirrors the third-party WearECG public VAE
implementation as closely as possible. The main local additions are:
- ASCII comments/docstrings
- a small compatibility wrapper (`WearECGVAE`) for local evaluators

The exact baseline training path should use `VAE_Encoder`, `VAE_Decoder`, and
`loss_function` directly.
"""

from __future__ import annotations

import math

import torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import kl_divergence
from torch.distributions.normal import Normal


LATENT_SCALE = 0.18215


class BaselineSelfAttention(nn.Module):
    def __init__(self, n_heads: int, d_embed: int, in_proj_bias=True, out_proj_bias=True):
        super().__init__()
        self.in_proj = nn.Linear(d_embed, 3 * d_embed, bias=in_proj_bias)
        self.out_proj = nn.Linear(d_embed, d_embed, bias=out_proj_bias)
        self.n_heads = n_heads
        self.d_head = d_embed // n_heads

    def forward(self, x: torch.Tensor, causal_mask=False):
        input_shape = x.shape
        batch_size, sequence_length, _d_embed = input_shape
        interim_shape = (batch_size, sequence_length, self.n_heads, self.d_head)

        q, k, v = self.in_proj(x).chunk(3, dim=-1)

        q = q.view(interim_shape).transpose(1, 2)
        k = k.view(interim_shape).transpose(1, 2)
        v = v.view(interim_shape).transpose(1, 2)

        weight = q @ k.transpose(-1, -2)

        if causal_mask:
            mask = torch.ones_like(weight, dtype=torch.bool).triu(1)
            weight.masked_fill_(mask, -torch.inf)

        weight /= math.sqrt(self.d_head)
        weight = F.softmax(weight, dim=-1)

        output = weight @ v
        output = output.transpose(1, 2)
        output = output.reshape(input_shape)
        output = self.out_proj(output)
        return output


class BaselineAttentionBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.groupnorm = nn.GroupNorm(32, channels)
        self.attention = SelfAttention(1, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x
        x = x.transpose(-1, -2)
        x = self.attention(x)
        x = x.transpose(-1, -2)
        x += residue
        return x


class BaselineResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.groupnorm_1 = nn.GroupNorm(32, in_channels)
        self.conv_1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)

        self.groupnorm_2 = nn.GroupNorm(32, out_channels)
        self.conv_2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            self.residual_layer = nn.Conv1d(in_channels, out_channels, kernel_size=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x

        x = self.groupnorm_1(x)
        x = F.silu(x)
        x = self.conv_1(x)

        x = self.groupnorm_2(x)
        x = F.silu(x)
        x = self.conv_2(x)

        return x + self.residual_layer(residue)


class BaselineEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(12, 128, kernel_size=3, padding=1),
                VAE_ResidualBlock(128, 128),
                VAE_ResidualBlock(128, 128),
                nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(128, 256),
                VAE_ResidualBlock(256, 256),
                nn.Conv1d(256, 256, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(256, 512),
                VAE_ResidualBlock(512, 512),
                nn.Conv1d(512, 512, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),
                nn.GroupNorm(32, 512),
                nn.SiLU(),
                nn.Conv1d(512, 8, kernel_size=3, padding=1),
                nn.Conv1d(8, 8, kernel_size=1, padding=0),
            ]
        )

    def forward(self, x: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
        x = x.transpose(1, 2)

        for module in self.blocks:
            if getattr(module, "stride", None) == (2,):
                x = F.pad(x, (0, 1))
            x = module(x)

        mean, log_variance = torch.chunk(x, 2, dim=1)
        log_variance = torch.clamp(log_variance, -30, 20)
        variance = log_variance.exp()
        stdev = variance.sqrt()

        if noise is None:
            noise = torch.randn(stdev.shape, device=stdev.device)
        x = mean + stdev * noise
        x *= LATENT_SCALE

        return x, mean, log_variance


class BaselineDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(4, 4, kernel_size=1, padding=0),
                nn.Conv1d(4, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 256),
                VAE_ResidualBlock(256, 256),
                VAE_ResidualBlock(256, 256),
                nn.GroupNorm(32, 256),
                nn.SiLU(),
                nn.Conv1d(256, 12, kernel_size=3, padding=1),
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x /= LATENT_SCALE

        for module in self.blocks:
            x = module(x)

        x = x.transpose(1, 2)
        return x


def loss_function(
    recons,
    x,
    mu,
    log_var,
    kld_weight=1e-4,
    perceptual_weight=1e-2,
    ecg_founder_model=None,
    device=None,
) -> dict:
    recons_loss = F.mse_loss(recons, x, reduction="mean")

    q_z_x = Normal(mu, log_var.mul(0.5).exp())
    p_z = Normal(torch.zeros_like(mu), torch.ones_like(log_var))
    kld_loss = kl_divergence(q_z_x, p_z).sum(1).mean()
    loss = recons_loss + kld_weight * kld_loss

    return {
        "loss": loss,
        "recons_loss": recons_loss.detach(),
        "KLD_loss": kld_loss.detach(),
        "perceptual_loss": torch.tensor(0.0, device=mu.device),
    }


class BaselineWearECGVAE(nn.Module):
    """Compatibility wrapper for local evaluation code.

    This wrapper preserves the exact public modules while exposing the minimal
    local `stage1`/`impute_from_regressor` API expected by the shared evaluator.
    """

    def __init__(self, in_channels: int = 12, out_channels: int = 12, latent_channels: int = 4, target_len: int = 5000, beta_kl: float = 1e-4, chest_weighted: bool = False):
        super().__init__()
        if in_channels != 12 or out_channels != 12:
            raise ValueError("WearECGVAE expects 12-lead input/output.")
        if latent_channels != 4:
            raise ValueError("Exact WearECG VAE baseline uses latent_channels=4.")
        self.target_len = int(target_len)
        self.beta_kl = float(beta_kl)
        self.chest_weighted = bool(chest_weighted)
        self.encoder = BaselineEncoder()
        self.decoder = BaselineDecoder()

    def _match_target_len(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] < self.target_len:
            x = F.pad(x, (0, 0, 0, self.target_len - x.shape[1]))
        elif x.shape[1] > self.target_len:
            x = x[:, : self.target_len, :]
        return x

    def stage1_forward(self, x: torch.Tensor, lead_indices=None):
        x_seq = x.transpose(1, 2)
        z, mean, log_var = self.encoder(x_seq)
        recons = self._match_target_len(self.decoder(z))
        loss = loss_function(recons, x_seq, mean, log_var, kld_weight=self.beta_kl)
        y_pred = recons.transpose(1, 2)
        zero = torch.tensor(0.0, device=x.device)
        return {
            "loss": loss["loss"],
            "decoder_loss": loss["recons_loss"],
            "teacher_loss": zero,
            "align_loss": zero,
            "stft_loss": zero,
            "diff_loss": zero,
            "corr_loss": zero,
            "kl_loss": loss["KLD_loss"],
            "y_target": x,
            "y_pred": y_pred,
            "y_pred_reg": y_pred,
            "z_regressed": mean.detach(),
        }

    @torch.no_grad()
    def impute_from_regressor(self, x: torch.Tensor, lead_indices=None):
        x_seq = x.transpose(1, 2)
        _z, mean, _log_var = self.encoder(x_seq)
        recons = self._match_target_len(self.decoder(mean * LATENT_SCALE))
        return {
            "available": True,
            "y_pred": recons.transpose(1, 2),
            "z_latent": mean.detach(),
        }

    def forward(self, x: torch.Tensor, lead_indices=None, mode: str = "stage1", **kwargs):
        if mode != "stage1":
            raise ValueError("WearECGVAE supports only mode='stage1'.")
        return self.stage1_forward(x, lead_indices=lead_indices)


### 8B. FM-Integrated WearECG (vae_fm.py)


In [7]:
#!/usr/bin/env python3
"""
WearECG + ECG-FM integration.

Design goal:
- Preserve the baseline WearECG VAE geometry and reconstruction path.
- Use ECG-FM as a frozen semantic teacher first.
- Optionally use pooled ECG-FM context to condition the decoder.
- Keep the external API compatible with the existing evaluator.

Input contract:
- public wrappers expect x shaped [B, 12, T], where x is a sparse 12-lead canvas
- y_full, when provided, is the full 12-lead target shaped [B, 12, T]
- internal sequence conversions stay local to the model

Main classes:
- WearECGVAE: exact-ish baseline wrapper
- WearECGFMVAE: baseline + frozen ECG-FM perceptual supervision
"""

from __future__ import annotations

import math
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import kl_divergence
from torch.distributions.normal import Normal

LATENT_SCALE = 0.18215


# =============================================================================
# FAIRSEQ / ECG-FM LOADING
# =============================================================================

def _setup_fairseq_path() -> None:
    """Best-effort addition of fairseq-signals to sys.path."""
    try:
        if '__file__' in globals():
            project_root = Path(__file__).resolve().parent
            candidates = [
                project_root / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent.parent.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent.parent.parent.parent / "ecg_fm_integration" / "fairseq-signals",
            ]
        else:
            # Notebook context or fallback
            candidates = [
                Path("/home/mithunmanivannan/ecg_fm_integration/fairseq-signals"),
                Path("/home/mithunmanivannan/third_party/WearECG-reconstruction/ecg_fm_integration/fairseq-signals"),
            ]
            
        for fs_path in candidates:
            if fs_path.exists():
                fs_path_str = str(fs_path)
                if fs_path_str not in sys.path:
                    sys.path.insert(0, fs_path_str)
                break
    except Exception:
        pass


_setup_fairseq_path()


class ECGFMFeatureExtractor(nn.Module):
    """
    Frozen ECG-FM feature extractor.

    Returns token embeddings of shape [B, T_enc, D].
    Also provides simple pooled summaries for perceptual supervision / conditioning.
    """

    def __init__(
        self,
        checkpoint_path: str,
        embed_dim: int = 768,
        allow_unexpected_keys: Optional[List[str]] = None,
    ) -> None:
        super().__init__()
        self.embed_dim = int(embed_dim)
        self.allow_unexpected_keys = set(allow_unexpected_keys or [])

        from fairseq_signals.models.ecg_transformer import ECGTransformerModel
        from fairseq_signals.utils.checkpoint_utils import load_checkpoint_to_cpu
        from omegaconf import OmegaConf

        state = load_checkpoint_to_cpu(checkpoint_path)
        cfg = state["cfg"]["model"]
        OmegaConf.set_struct(cfg, False)
        if getattr(cfg, "saliency", None) is None:
            cfg.saliency = False

        self.backbone = ECGTransformerModel.build_model(cfg)
        incompatible = self.backbone.load_state_dict(state["model"], strict=False)

        if incompatible.missing_keys:
            raise RuntimeError(f"ECG-FM checkpoint missing keys: {incompatible.missing_keys}")

        unexpected = set(incompatible.unexpected_keys)
        disallowed = sorted(unexpected - self.allow_unexpected_keys)
        if disallowed:
            raise RuntimeError(f"ECG-FM checkpoint has unsupported unexpected keys: {disallowed}")

        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()

        self.token_norm = nn.LayerNorm(self.embed_dim)

    def train(self, mode: bool = True):
        super().train(mode)
        # The FM is a frozen teacher/conditioner and should never leave eval mode.
        self.backbone.eval()
        return self

    def extract_tokens(self, x_12: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x_12: [B, 12, T], float tensor in ECG units

        Returns:
            tokens: [B, T_enc, D]
        """
        self.backbone.eval()
        x_norm = self._full12_zscore(x_12).to(torch.float32)
        device_type = x_norm.device.type
        with torch.amp.autocast(device_type, enabled=False):
            res = self.backbone.extract_features(x_norm, None)
            tokens = res["x"] if isinstance(res, dict) else res
            tokens = self.token_norm(tokens)
        return tokens.to(x_12.dtype)

    @staticmethod
    def _full12_zscore(x_12: torch.Tensor) -> torch.Tensor:
        """
        Per-lead z-score over time for fully observed 12-lead ECG.
        """
        mean = x_12.mean(dim=2, keepdim=True)
        std = x_12.std(dim=2, keepdim=True).clamp(min=1e-6)
        return (x_12 - mean) / std

    @staticmethod
    def pooled_summary(tokens: torch.Tensor) -> torch.Tensor:
        """
        Global pooled summary: [B, D]
        """
        return tokens.mean(dim=1)


# =============================================================================
# EXACT / BASELINE WEARECG BLOCKS
# =============================================================================

class SelfAttention(nn.Module):
    def __init__(self, n_heads: int, d_embed: int, in_proj_bias: bool = True, out_proj_bias: bool = True):
        super().__init__()
        self.in_proj = nn.Linear(d_embed, 3 * d_embed, bias=in_proj_bias)
        self.out_proj = nn.Linear(d_embed, d_embed, bias=out_proj_bias)
        self.n_heads = n_heads
        self.d_head = d_embed // n_heads

    def forward(self, x: torch.Tensor, causal_mask: bool = False) -> torch.Tensor:
        input_shape = x.shape
        batch_size, sequence_length, _d_embed = input_shape
        interim_shape = (batch_size, sequence_length, self.n_heads, self.d_head)

        q, k, v = self.in_proj(x).chunk(3, dim=-1)

        q = q.view(interim_shape).transpose(1, 2)
        k = k.view(interim_shape).transpose(1, 2)
        v = v.view(interim_shape).transpose(1, 2)

        weight = q @ k.transpose(-1, -2)

        if causal_mask:
            mask = torch.ones_like(weight, dtype=torch.bool).triu(1)
            weight.masked_fill_(mask, -torch.inf)

        weight = weight / math.sqrt(self.d_head)
        weight = F.softmax(weight, dim=-1)

        output = weight @ v
        output = output.transpose(1, 2)
        output = output.reshape(input_shape)
        output = self.out_proj(output)
        return output


class VAE_AttentionBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.groupnorm = nn.GroupNorm(32, channels)
        self.attention = SelfAttention(1, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x
        x = x.transpose(-1, -2)
        x = self.attention(x)
        x = x.transpose(-1, -2)
        return x + residue


class VAE_ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.groupnorm_1 = nn.GroupNorm(32, in_channels)
        self.conv_1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)

        self.groupnorm_2 = nn.GroupNorm(32, out_channels)
        self.conv_2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            self.residual_layer = nn.Conv1d(in_channels, out_channels, kernel_size=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x

        x = self.groupnorm_1(x)
        x = F.silu(x)
        x = self.conv_1(x)

        x = self.groupnorm_2(x)
        x = F.silu(x)
        x = self.conv_2(x)

        return x + self.residual_layer(residue)


class VAE_Encoder(nn.Module):
    """
    Near-verbatim WearECG encoder.
    Expects x shaped [B, 12, T] or [B, T, 12] if called through wrapper.
    """

    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(12, 128, kernel_size=3, padding=1),
                VAE_ResidualBlock(128, 128),
                VAE_ResidualBlock(128, 128),
                nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(128, 256),
                VAE_ResidualBlock(256, 256),
                nn.Conv1d(256, 256, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(256, 512),
                VAE_ResidualBlock(512, 512),
                nn.Conv1d(512, 512, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),
                nn.GroupNorm(32, 512),
                nn.SiLU(),
                nn.Conv1d(512, 8, kernel_size=3, padding=1),
                nn.Conv1d(8, 8, kernel_size=1, padding=0),
            ]
        )

    def forward(self, x: torch.Tensor, noise: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # If caller passed [B, T, 12], convert to [B, 12, T]
        if x.dim() != 3:
            raise ValueError(f"Expected rank-3 input, got shape {tuple(x.shape)}")
        if x.shape[1] != 12 and x.shape[2] == 12:
            x = x.transpose(1, 2)
        elif x.shape[1] != 12:
            raise ValueError(f"Expected 12 channels, got shape {tuple(x.shape)}")

        for module in self.blocks:
            if getattr(module, "stride", None) == (2,):
                x = F.pad(x, (0, 1))
            x = module(x)

        mean, log_variance = torch.chunk(x, 2, dim=1)
        log_variance = torch.clamp(log_variance, -30, 20)
        variance = log_variance.exp()
        stdev = variance.sqrt()

        if noise is None:
            noise = torch.randn_like(stdev)
        z = mean + stdev * noise
        z = z * LATENT_SCALE

        return z, mean, log_variance


class ConditionalVAE_Decoder(nn.Module):
    """
    WearECG decoder geometry with optional pooled-FM FiLM conditioning.

    Conditioning is deliberately low-risk:
    - pooled ECG-FM summary only
    - applied after residual blocks
    - zero-initialized modulation so decoder starts as baseline
    """

    def __init__(self, cond_dim: int = 0):
        super().__init__()
        self.cond_dim = int(cond_dim)

        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(4, 4, kernel_size=1, padding=0),
                nn.Conv1d(4, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),   # 0
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),   # 1
                VAE_ResidualBlock(512, 512),   # 2
                VAE_ResidualBlock(512, 512),   # 3
                VAE_ResidualBlock(512, 512),   # 4
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),   # 5
                VAE_ResidualBlock(512, 512),   # 6
                VAE_ResidualBlock(512, 512),   # 7
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),   # 8
                VAE_ResidualBlock(512, 512),   # 9
                VAE_ResidualBlock(512, 512),   # 10
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 256),   # 11
                VAE_ResidualBlock(256, 256),   # 12
                VAE_ResidualBlock(256, 256),   # 13
                nn.GroupNorm(32, 256),
                nn.SiLU(),
                nn.Conv1d(256, 12, kernel_size=3, padding=1),
            ]
        )

        residual_out_channels = [
            512, 512, 512, 512, 512,
            512, 512, 512,
            512, 512, 512,
            256, 256, 256,
        ]

        if self.cond_dim > 0:
            self.cond_to_film = nn.ModuleList(
                [nn.Linear(self.cond_dim, 2 * c) for c in residual_out_channels]
            )
            for layer in self.cond_to_film:
                nn.init.zeros_(layer.weight)
                nn.init.zeros_(layer.bias)
        else:
            self.cond_to_film = nn.ModuleList()

    def _apply_film(self, x: torch.Tensor, cond_vec: Optional[torch.Tensor], film_idx: int) -> torch.Tensor:
        if cond_vec is None or self.cond_dim == 0:
            return x
        gamma_beta = self.cond_to_film[film_idx](cond_vec)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        gamma = gamma.unsqueeze(-1)
        beta = beta.unsqueeze(-1)
        return x * (1.0 + gamma) + beta

    def forward(self, z: torch.Tensor, cond_vec: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = z / LATENT_SCALE
        film_idx = 0

        for module in self.blocks:
            x = module(x)
            if isinstance(module, VAE_ResidualBlock):
                x = self._apply_film(x, cond_vec, film_idx)
                film_idx += 1

        x = x.transpose(1, 2)
        return x


# =============================================================================
# LOSSES
# =============================================================================

def baseline_loss_function(
    recons: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    kld_weight: float = 1e-4,
    lead_indices: Optional[torch.Tensor] = None,
    missing_lead_weight: float = 1.0,
) -> Dict[str, torch.Tensor]:
    recons_loss = weighted_reconstruction_mse(
        recons,
        x,
        lead_indices=lead_indices,
        missing_lead_weight=missing_lead_weight,
    )

    q_z_x = Normal(mu, log_var.mul(0.5).exp())
    p_z = Normal(torch.zeros_like(mu), torch.ones_like(log_var))
    kld_loss = kl_divergence(q_z_x, p_z).sum(1).mean()

    loss = recons_loss + kld_weight * kld_loss

    return {
        "loss": loss,
        "recons_loss": recons_loss.detach(),
        "KLD_loss": kld_loss.detach(),
    }


def weighted_reconstruction_mse(
    recons: torch.Tensor,
    target: torch.Tensor,
    *,
    lead_indices: Optional[torch.Tensor] = None,
    missing_lead_weight: float = 1.0,
) -> torch.Tensor:
    """Weighted full-12 reconstruction MSE.

    Defaults to the exact baseline behavior when ``missing_lead_weight == 1.0``.
    When increased, only missing leads receive extra weight, while observed leads
    remain weight 1.0. This keeps the baseline comparison path intact while
    allowing a sparse-reconstruction emphasis ablation.
    """
    if lead_indices is None or float(missing_lead_weight) == 1.0:
        return F.mse_loss(recons, target, reduction="mean")

    if lead_indices.dim() != 2:
        raise ValueError(f"Expected lead_indices shape [B, N_obs], got {tuple(lead_indices.shape)}")
    if recons.dim() != 3 or target.dim() != 3:
        raise ValueError("Expected reconstruction and target tensors shaped [B, T, 12].")
    if recons.shape != target.shape:
        raise ValueError(f"Expected matching reconstruction/target shapes, got {tuple(recons.shape)} vs {tuple(target.shape)}")

    batch_size, _, num_leads = recons.shape
    weights = recons.new_full((batch_size, num_leads), float(missing_lead_weight))
    observed_weight = recons.new_ones((batch_size, lead_indices.shape[1]))
    weights.scatter_(1, lead_indices.long(), observed_weight)
    weights = weights.unsqueeze(1)
    squared_error = (recons - target) ** 2
    time_len = recons.shape[1]
    return (squared_error * weights).sum() / (weights.sum().clamp(min=1.0) * time_len)


def pooled_fm_perceptual_loss(
    fm_model: ECGFMFeatureExtractor,
    x_recon_12: torch.Tensor,
    pooled_true: torch.Tensor,
    *,
    cosine_mix: float = 0.5,
) -> torch.Tensor:
    """
    Pooled FM perceptual loss:
    - MSE in pooled ECG-FM space
    - optional cosine component

    Args:
        x_recon_12: [B, 12, T]
    """
    recon_tokens = fm_model.extract_tokens(x_recon_12)
    recon_pool = fm_model.pooled_summary(recon_tokens)

    mse_term = F.mse_loss(recon_pool, pooled_true, reduction="mean")
    cos_term = 1.0 - F.cosine_similarity(recon_pool, pooled_true, dim=1).mean()

    return (1.0 - cosine_mix) * mse_term + cosine_mix * cos_term


# =============================================================================
# EXACT BASELINE WRAPPER
# =============================================================================

class WearECGVAE(nn.Module):
    """
    Exact-ish baseline compatibility wrapper.

    External API:
    - stage1_forward
    - impute_from_regressor
    - forward(..., mode='stage1')
    """

    def __init__(
        self,
        in_channels: int = 12,
        out_channels: int = 12,
        latent_channels: int = 4,
        target_len: int = 5000,
        beta_kl: float = 1e-4,
        chest_weighted: bool = False,
        missing_lead_weight: float = 1.0,
    ):
        super().__init__()
        if in_channels != 12 or out_channels != 12:
            raise ValueError("WearECGVAE expects 12-lead input/output.")
        if latent_channels != 4:
            raise ValueError("Exact WearECG VAE baseline uses latent_channels=4.")

        self.target_len = int(target_len)
        self.beta_kl = float(beta_kl)
        self.chest_weighted = bool(chest_weighted)
        self.missing_lead_weight = float(missing_lead_weight)

        self.encoder = VAE_Encoder()
        self.decoder = ConditionalVAE_Decoder(cond_dim=0)

    def _match_target_len(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] < self.target_len:
            x = F.pad(x, (0, 0, 0, self.target_len - x.shape[1]))
        elif x.shape[1] > self.target_len:
            x = x[:, : self.target_len, :]
        return x

    def stage1_forward(
        self,
        x: torch.Tensor,
        y_full: Optional[torch.Tensor] = None,
        lead_indices=None,
    ) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")
        if y_full is not None and (y_full.dim() != 3 or y_full.shape[1] != 12):
            raise ValueError(f"Expected y_full shape [B, 12, T], got {tuple(y_full.shape)}")

        target_12 = y_full if y_full is not None else x
        z, mean, log_var = self.encoder(x)
        recons = self._match_target_len(self.decoder(z, cond_vec=None))
        y_pred = recons.transpose(1, 2)
        loss = baseline_loss_function(
            recons,
            target_12.transpose(1, 2),
            mean,
            log_var,
            kld_weight=self.beta_kl,
            lead_indices=lead_indices,
            missing_lead_weight=self.missing_lead_weight,
        )

        zero = torch.tensor(0.0, device=x.device)
        return {
            "loss": loss["loss"],
            "decoder_loss": loss["recons_loss"],
            "teacher_loss": zero,
            "align_loss": zero,
            "stft_loss": zero,
            "diff_loss": zero,
            "corr_loss": zero,
            "kl_loss": loss["KLD_loss"],
            "y_target": target_12,
            "y_pred": y_pred,
            "y_pred_reg": y_pred,
            "z_regressed": mean.detach(),
        }

    @torch.no_grad()
    def impute_from_regressor(self, x: torch.Tensor, lead_indices=None) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")

        _z, mean, _log_var = self.encoder(x)
        recons = self._match_target_len(self.decoder(mean * LATENT_SCALE, cond_vec=None))
        return {
            "available": True,
            "y_pred": recons.transpose(1, 2),
            "z_latent": mean.detach(),
        }

    def forward(self, x: torch.Tensor, lead_indices=None, mode: str = "stage1", **kwargs):
        if mode != "stage1":
            raise ValueError("WearECGVAE supports only mode='stage1'.")
        return self.stage1_forward(x, lead_indices=lead_indices, y_full=kwargs.get("y_full"))


# =============================================================================
# FM-INTEGRATED WRAPPER
# =============================================================================

class WearECGFMVAE(nn.Module):
    """
    WearECG baseline + ECG-FM integration.

    Integration strategy:
    1. Preserve the native WearECG encoder-decoder path.
    2. Add frozen ECG-FM perceptual supervision on reconstructions.
    3. Optionally add pooled ECG-FM decoder conditioning.

    External API matches the baseline wrapper.
    """

    def __init__(
        self,
        fm_checkpoint_path: str,
        in_channels: int = 12,
        out_channels: int = 12,
        latent_channels: int = 4,
        target_len: int = 5000,
        beta_kl: float = 1e-4,
        chest_weighted: bool = False,
        missing_lead_weight: float = 1.0,
        fm_embed_dim: int = 768,
        fm_loss_weight: float = 1e-2,
        fm_cosine_mix: float = 0.5,
        use_decoder_conditioning: bool = False,
        fm_cond_drop_prob: float = 0.0,
        use_latent_alignment: bool = False,
        latent_align_weight: float = 1e-3,
    ):
        super().__init__()
        if in_channels != 12 or out_channels != 12:
            raise ValueError("WearECGFMVAE expects 12-lead input/output.")
        if latent_channels != 4:
            raise ValueError("This implementation preserves the exact WearECG latent_channels=4 baseline.")

        self.target_len = int(target_len)
        self.beta_kl = float(beta_kl)
        self.chest_weighted = bool(chest_weighted)
        self.missing_lead_weight = float(missing_lead_weight)

        self.fm_loss_weight = float(fm_loss_weight)
        self.fm_cosine_mix = float(fm_cosine_mix)
        self.use_decoder_conditioning = bool(use_decoder_conditioning)
        self.fm_cond_drop_prob = float(fm_cond_drop_prob)
        self.use_latent_alignment = bool(use_latent_alignment)
        self.latent_align_weight = float(latent_align_weight)
        self.fm_embed_dim = int(fm_embed_dim)

        self.encoder = VAE_Encoder()

        self.fm_model = ECGFMFeatureExtractor(
            checkpoint_path=fm_checkpoint_path,
            embed_dim=fm_embed_dim,
            allow_unexpected_keys=[
                "quantizer.vars",
                "quantizer.weight_proj.weight",
                "quantizer.weight_proj.bias",
                "project_q.weight",
                "project_q.bias",
                "final_proj.weight",
                "final_proj.bias",
            ],
        )

        cond_dim = fm_embed_dim if self.use_decoder_conditioning else 0
        self.decoder = ConditionalVAE_Decoder(cond_dim=cond_dim)
        self.latent_projector = nn.Sequential(
            nn.Linear(4, 256),
            nn.SiLU(),
            nn.Linear(256, self.fm_embed_dim),
        )

    def _match_target_len(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] < self.target_len:
            x = F.pad(x, (0, 0, 0, self.target_len - x.shape[1]))
        elif x.shape[1] > self.target_len:
            x = x[:, : self.target_len, :]
        return x

    def _get_decoder_condition(self, x_12: torch.Tensor) -> Optional[torch.Tensor]:
        if not self.use_decoder_conditioning:
            return None

        with torch.no_grad():
            tokens = self.fm_model.extract_tokens(x_12)
            cond_vec = self.fm_model.pooled_summary(tokens)

        if self.training and self.fm_cond_drop_prob > 0.0:
            keep = (torch.rand(cond_vec.size(0), device=cond_vec.device) > self.fm_cond_drop_prob).float()
            cond_vec = cond_vec * keep.unsqueeze(1)

        return cond_vec.to(x_12.dtype)

    def _latent_align_loss(self, mean: torch.Tensor, pooled_true: torch.Tensor) -> torch.Tensor:
        if not self.use_latent_alignment:
            return mean.new_tensor(0.0)
        mean_pool = mean.float().mean(dim=2)
        pred = self.latent_projector(mean_pool)
        return F.mse_loss(pred, pooled_true.detach().float(), reduction="mean")

    def stage1_forward(
        self,
        x: torch.Tensor,
        y_full: Optional[torch.Tensor] = None,
        lead_indices=None,
    ) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")
        if y_full is not None and (y_full.dim() != 3 or y_full.shape[1] != 12):
            raise ValueError(f"Expected y_full shape [B, 12, T], got {tuple(y_full.shape)}")

        target_12 = y_full if y_full is not None else x
        # Baseline path
        z, mean, log_var = self.encoder(x)

        cond_vec = self._get_decoder_condition(x)
        recons = self._match_target_len(self.decoder(z, cond_vec=cond_vec))
        y_pred = recons.transpose(1, 2)

        base_loss = baseline_loss_function(
            recons,
            target_12.transpose(1, 2),
            mean,
            log_var,
            kld_weight=self.beta_kl,
            lead_indices=lead_indices,
            missing_lead_weight=self.missing_lead_weight,
        )

        # FM perceptual supervision
        with torch.no_grad():
            true_tokens = self.fm_model.extract_tokens(target_12)
            pooled_true = self.fm_model.pooled_summary(true_tokens)

        fm_teacher = pooled_fm_perceptual_loss(
            fm_model=self.fm_model,
            x_recon_12=y_pred,
            pooled_true=pooled_true.detach(),
            cosine_mix=self.fm_cosine_mix,
        )
        latent_align = self._latent_align_loss(mean, pooled_true)

        total_loss = base_loss["loss"] + self.fm_loss_weight * fm_teacher + self.latent_align_weight * latent_align
        zero = x.new_tensor(0.0)

        return {
            "loss": total_loss,
            "decoder_loss": base_loss["recons_loss"],
            "teacher_loss": fm_teacher.detach(),
            "align_loss": latent_align.detach() if self.use_latent_alignment else zero,
            "stft_loss": zero,
            "diff_loss": zero,
            "corr_loss": zero,
            "kl_loss": base_loss["KLD_loss"],
            "y_target": target_12,
            "y_pred": y_pred,
            "y_pred_reg": y_pred,
            "z_regressed": mean.detach(),
            "fm_perceptual_loss": fm_teacher.detach(),
            "latent_align_loss": latent_align.detach() if self.use_latent_alignment else zero,
        }

    @torch.no_grad()
    def impute_from_regressor(self, x: torch.Tensor, lead_indices=None) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")

        _z, mean, _log_var = self.encoder(x)

        cond_vec = self._get_decoder_condition(x)
        recons = self._match_target_len(self.decoder(mean * LATENT_SCALE, cond_vec=cond_vec))

        return {
            "available": True,
            "y_pred": recons.transpose(1, 2),
            "z_latent": mean.detach(),
        }

    def forward(self, x: torch.Tensor, lead_indices=None, mode: str = "stage1", **kwargs):
        if mode != "stage1":
            raise ValueError("WearECGFMVAE supports only mode='stage1'.")
        return self.stage1_forward(x, lead_indices=lead_indices, y_full=kwargs.get("y_full"))


## 9. Sunnybrook Clinical Evaluation (Baseline WearECG)

Zero-shot clinical evaluation on Sunnybrook Sierra XMLs using the baseline WearECG VAE.
Includes waveform fidelity and robust clinical morphology via NeuroKit2.


In [8]:
import sierraecg
import neurokit2 as nk
from scipy.signal import butter, filtfilt
from tqdm import tqdm

LEAD_ORDER = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
EINTHOVEN_NOISE_FLOOR_UV = 389
AUGMENTED_NOISE_FLOOR_UV = 296

def normalize_mason(sig_mv):
    # Baseline VAE normalization: Clip [-2.5, 2.5] and shift to [0, 1]
    return np.clip((sig_mv + 2.5) / 5.0, 0, 1)

def denormalize_mason(sig_norm):
    return sig_norm * 5.0 - 2.5

def load_sunnybrook_record_v2(xml_path, target_len=5000):
    """Load Sunnybrook record with no legacy dependencies."""
    try:
        f = sierraecg.read_file(str(xml_path))
        signal_map = {lead.label: lead.samples for lead in f.leads}
        if not all(l in signal_map for l in LEAD_ORDER):
            return None, None
        sig = np.stack([signal_map[l] for l in LEAD_ORDER]).astype(np.float64)
        sig_mv = sig / 1000.0
        curr_len = sig_mv.shape[1]
        # Crop or Pad
        if curr_len < target_len:
            sig_mv = np.concatenate([sig_mv, np.zeros((12, target_len - curr_len))], axis=1)
        else:
            sig_mv = sig_mv[:, :target_len]
        sig_norm = normalize_mason(sig_mv)
        return torch.tensor(sig_norm, dtype=torch.float32), sig_mv
    except Exception as e:
        return None, None

def compute_summary_metrics(recon_mv, gt_mv):
    """Waveform and Physics metrics."""
    mse = F.mse_loss(recon_mv, gt_mv).item()
    pearson = []
    for i in range(12):
        r, g = recon_mv[0, i], gt_mv[0, i]
        r_cent, g_cent = r - r.mean(), g - g.mean()
        num = (r_cent * g_cent).sum()
        den = torch.sqrt((r_cent ** 2).sum() * (g_cent ** 2).sum() + 1e-8)
        pearson.append((num / den).item())
    
    # Physics Gates
    r = recon_mv[0]
    e_res = r[1] - (r[0] + r[2]) # II - (I + III)
    e_rms_uv = torch.sqrt(torch.mean(e_res ** 2)).item() * 1000.0
    a_res = r[3] + (r[0] + r[1]) / 2.0 # aVR + (I + II) / 2
    a_rms_uv = torch.sqrt(torch.mean(a_res ** 2)).item() * 1000.0
    
    return {
        "mse_mv": mse, 
        "mean_pearson": np.mean(pearson),
        "e_rms_uv": e_rms_uv,
        "a_rms_uv": a_rms_uv
    }

def extract_morphology(sig_mv, fs=500):
    """NeuroKit2 morphology extraction."""
    try:
        # Lead II is usually best for global morphology
        ecg_signals, info = nk.ecg_process(sig_mv[1], sampling_rate=fs)
        hr = ecg_signals['ECG_Rate'].median()
        rpeaks = info['ECG_R_Peaks']
        
        # Basic checks
        if len(rpeaks) < 2: return {}
        
        # Delineate for P, Q, S, T
        try:
            waves, _ = nk.ecg_delineate(ecg_signals['ECG_Clean'], rpeaks, sampling_rate=fs)
            qrs_dur = (np.nanmedian(waves['ECG_S_Offsets']) - np.nanmedian(waves['ECG_Q_Offsets'])) / fs * 1000
            qt_dur = (np.nanmedian(waves['ECG_T_Offsets']) - np.nanmedian(waves['ECG_R_Onsets'])) / fs * 1000
            # QRS Axis (I and aVF)
            net_i = np.mean(sig_mv[0][rpeaks])
            net_avf = np.mean(sig_mv[5][rpeaks])
            axis = np.degrees(np.arctan2(net_avf, net_i))
            
            return {"hr": hr, "qrs": qrs_dur, "qt": qt_dur, "axis": axis}
        except: return {"hr": hr}
    except: return {}
print("Robust evaluation suite initialized.")


Robust evaluation suite initialized.


In [9]:
BASELINE_CKPT = "/home/mithunmanivannan/checkpoints/ul_ecg/engineering_wearecg_exact_II-V1-V5_bs80_lf1.5/ul_ecp_best.pt"
SUNNYBROOK_ROOT = Path("/home/mithunmanivannan/data/sunnybrook")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if BASELINE_CKPT and SUNNYBROOK_ROOT.exists():
    model = BaselineWearECGVAE().to(DEVICE)
    ckpt = torch.load(BASELINE_CKPT, map_location=DEVICE)
    model.encoder.load_state_dict(ckpt['encoder_state_dict'])
    model.decoder.load_state_dict(ckpt['decoder_state_dict'])
    model.eval()
    
    obs_leads = [1, 6, 10] # II, V1, V5
    xmls = sorted(list(SUNNYBROOK_ROOT.glob("*.xml")))[:5]
    print(f"Zero-shot eval on {len(xmls)} Sunnybrook records...")
    
    results = []
    for path in xmls:
        sig_norm, sig_mv = load_sunnybrook_record_v2(path)
        if sig_norm is not None:
            x = torch.zeros(1, 12, 5000, device=DEVICE)
            x[0, obs_leads] = sig_norm[obs_leads].to(DEVICE)
            
            with torch.no_grad():
                out = model.impute_from_regressor(x)
                pred_mv = out['y_pred']
            
            gt_mv = torch.tensor(sig_mv).unsqueeze(0).to(DEVICE)
            metrics = compute_summary_metrics(pred_mv, gt_mv)
            morph = extract_morphology(pred_mv[0].cpu().numpy())
            
            results.append({"file": path.name, **metrics, **morph})
    
    display(pd.DataFrame(results))
else:
    print("Prerequisites missing.")


Zero-shot eval on 5 Sunnybrook records...


,file,mse_mv,mean_pearson,e_rms_uv,a_rms_uv,hr
0,ECG001.xml,0.012210,0.244422,23.013460,27.424168,42.051885
1,ECG002.xml,0.013174,0.141040,26.598169,27.287737,93.212237
2,ECG003.xml,0.008871,0.101463,23.841113,26.664490,60.595081
3,ECG004.xml,0.006191,0.217905,26.707003,25.277855,106.555993
4,ECG005.xml,0.012997,0.086600,25.243836,25.423229,51.533742


## 10. Main Execution Context


In [10]:
FM_CKPT = "/home/mithunmanivannan/ecg_fm_integration/checkpoints/mimic_iv_ecg_physionet_pretrained.pt"
if os.path.exists(FM_CKPT):
    model_fm = WearECGFMVAE(fm_checkpoint_path=FM_CKPT).to(DEVICE)
    print("Active WearECG-FM model integrated (Ready for Stage 1 training).")


/home/mithunmanivannan/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active WearECG-FM model integrated (Ready for Stage 1 training).


## 11. Full Source Appendix


### Baseline WearECG VAE (vae.py)


In [11]:
"""Near-verbatim WearECG VAE modules and loss.

This file intentionally mirrors the third-party WearECG public VAE
implementation as closely as possible. The main local additions are:
- ASCII comments/docstrings
- a small compatibility wrapper (`WearECGVAE`) for local evaluators

The exact baseline training path should use `VAE_Encoder`, `VAE_Decoder`, and
`loss_function` directly.
"""

from __future__ import annotations

import math

import torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import kl_divergence
from torch.distributions.normal import Normal


LATENT_SCALE = 0.18215


class SelfAttention(nn.Module):
    def __init__(self, n_heads: int, d_embed: int, in_proj_bias=True, out_proj_bias=True):
        super().__init__()
        self.in_proj = nn.Linear(d_embed, 3 * d_embed, bias=in_proj_bias)
        self.out_proj = nn.Linear(d_embed, d_embed, bias=out_proj_bias)
        self.n_heads = n_heads
        self.d_head = d_embed // n_heads

    def forward(self, x: torch.Tensor, causal_mask=False):
        input_shape = x.shape
        batch_size, sequence_length, _d_embed = input_shape
        interim_shape = (batch_size, sequence_length, self.n_heads, self.d_head)

        q, k, v = self.in_proj(x).chunk(3, dim=-1)

        q = q.view(interim_shape).transpose(1, 2)
        k = k.view(interim_shape).transpose(1, 2)
        v = v.view(interim_shape).transpose(1, 2)

        weight = q @ k.transpose(-1, -2)

        if causal_mask:
            mask = torch.ones_like(weight, dtype=torch.bool).triu(1)
            weight.masked_fill_(mask, -torch.inf)

        weight /= math.sqrt(self.d_head)
        weight = F.softmax(weight, dim=-1)

        output = weight @ v
        output = output.transpose(1, 2)
        output = output.reshape(input_shape)
        output = self.out_proj(output)
        return output


class VAE_AttentionBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.groupnorm = nn.GroupNorm(32, channels)
        self.attention = SelfAttention(1, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x
        x = x.transpose(-1, -2)
        x = self.attention(x)
        x = x.transpose(-1, -2)
        x += residue
        return x


class VAE_ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.groupnorm_1 = nn.GroupNorm(32, in_channels)
        self.conv_1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)

        self.groupnorm_2 = nn.GroupNorm(32, out_channels)
        self.conv_2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            self.residual_layer = nn.Conv1d(in_channels, out_channels, kernel_size=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x

        x = self.groupnorm_1(x)
        x = F.silu(x)
        x = self.conv_1(x)

        x = self.groupnorm_2(x)
        x = F.silu(x)
        x = self.conv_2(x)

        return x + self.residual_layer(residue)


class VAE_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(12, 128, kernel_size=3, padding=1),
                VAE_ResidualBlock(128, 128),
                VAE_ResidualBlock(128, 128),
                nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(128, 256),
                VAE_ResidualBlock(256, 256),
                nn.Conv1d(256, 256, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(256, 512),
                VAE_ResidualBlock(512, 512),
                nn.Conv1d(512, 512, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),
                nn.GroupNorm(32, 512),
                nn.SiLU(),
                nn.Conv1d(512, 8, kernel_size=3, padding=1),
                nn.Conv1d(8, 8, kernel_size=1, padding=0),
            ]
        )

    def forward(self, x: torch.Tensor, noise: torch.Tensor = None) -> torch.Tensor:
        x = x.transpose(1, 2)

        for module in self.blocks:
            if getattr(module, "stride", None) == (2,):
                x = F.pad(x, (0, 1))
            x = module(x)

        mean, log_variance = torch.chunk(x, 2, dim=1)
        log_variance = torch.clamp(log_variance, -30, 20)
        variance = log_variance.exp()
        stdev = variance.sqrt()

        if noise is None:
            noise = torch.randn(stdev.shape, device=stdev.device)
        x = mean + stdev * noise
        x *= LATENT_SCALE

        return x, mean, log_variance


class VAE_Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(4, 4, kernel_size=1, padding=0),
                nn.Conv1d(4, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 256),
                VAE_ResidualBlock(256, 256),
                VAE_ResidualBlock(256, 256),
                nn.GroupNorm(32, 256),
                nn.SiLU(),
                nn.Conv1d(256, 12, kernel_size=3, padding=1),
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x /= LATENT_SCALE

        for module in self.blocks:
            x = module(x)

        x = x.transpose(1, 2)
        return x


def loss_function(
    recons,
    x,
    mu,
    log_var,
    kld_weight=1e-4,
    perceptual_weight=1e-2,
    ecg_founder_model=None,
    device=None,
) -> dict:
    recons_loss = F.mse_loss(recons, x, reduction="mean")

    q_z_x = Normal(mu, log_var.mul(0.5).exp())
    p_z = Normal(torch.zeros_like(mu), torch.ones_like(log_var))
    kld_loss = kl_divergence(q_z_x, p_z).sum(1).mean()
    loss = recons_loss + kld_weight * kld_loss

    return {
        "loss": loss,
        "recons_loss": recons_loss.detach(),
        "KLD_loss": kld_loss.detach(),
        "perceptual_loss": torch.tensor(0.0, device=mu.device),
    }


class WearECGVAE(nn.Module):
    """Compatibility wrapper for local evaluation code.

    This wrapper preserves the exact public modules while exposing the minimal
    local `stage1`/`impute_from_regressor` API expected by the shared evaluator.
    """

    def __init__(self, in_channels: int = 12, out_channels: int = 12, latent_channels: int = 4, target_len: int = 5000, beta_kl: float = 1e-4, chest_weighted: bool = False):
        super().__init__()
        if in_channels != 12 or out_channels != 12:
            raise ValueError("WearECGVAE expects 12-lead input/output.")
        if latent_channels != 4:
            raise ValueError("Exact WearECG VAE baseline uses latent_channels=4.")
        self.target_len = int(target_len)
        self.beta_kl = float(beta_kl)
        self.chest_weighted = bool(chest_weighted)
        self.encoder = VAE_Encoder()
        self.decoder = VAE_Decoder()

    def _match_target_len(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] < self.target_len:
            x = F.pad(x, (0, 0, 0, self.target_len - x.shape[1]))
        elif x.shape[1] > self.target_len:
            x = x[:, : self.target_len, :]
        return x

    def stage1_forward(self, x: torch.Tensor, lead_indices=None):
        x_seq = x.transpose(1, 2)
        z, mean, log_var = self.encoder(x_seq)
        recons = self._match_target_len(self.decoder(z))
        loss = loss_function(recons, x_seq, mean, log_var, kld_weight=self.beta_kl)
        y_pred = recons.transpose(1, 2)
        zero = torch.tensor(0.0, device=x.device)
        return {
            "loss": loss["loss"],
            "decoder_loss": loss["recons_loss"],
            "teacher_loss": zero,
            "align_loss": zero,
            "stft_loss": zero,
            "diff_loss": zero,
            "corr_loss": zero,
            "kl_loss": loss["KLD_loss"],
            "y_target": x,
            "y_pred": y_pred,
            "y_pred_reg": y_pred,
            "z_regressed": mean.detach(),
        }

    @torch.no_grad()
    def impute_from_regressor(self, x: torch.Tensor, lead_indices=None):
        x_seq = x.transpose(1, 2)
        _z, mean, _log_var = self.encoder(x_seq)
        recons = self._match_target_len(self.decoder(mean * LATENT_SCALE))
        return {
            "available": True,
            "y_pred": recons.transpose(1, 2),
            "z_latent": mean.detach(),
        }

    def forward(self, x: torch.Tensor, lead_indices=None, mode: str = "stage1", **kwargs):
        if mode != "stage1":
            raise ValueError("WearECGVAE supports only mode='stage1'.")
        return self.stage1_forward(x, lead_indices=lead_indices)


### Active FM-Integrated Model (vae_fm.py)


In [12]:
#!/usr/bin/env python3
"""
WearECG + ECG-FM integration.

Design goal:
- Preserve the baseline WearECG VAE geometry and reconstruction path.
- Use ECG-FM as a frozen semantic teacher first.
- Optionally use pooled ECG-FM context to condition the decoder.
- Keep the external API compatible with the existing evaluator.

Input contract:
- public wrappers expect x shaped [B, 12, T], where x is a sparse 12-lead canvas
- y_full, when provided, is the full 12-lead target shaped [B, 12, T]
- internal sequence conversions stay local to the model

Main classes:
- WearECGVAE: exact-ish baseline wrapper
- WearECGFMVAE: baseline + frozen ECG-FM perceptual supervision
"""

from __future__ import annotations

import math
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import kl_divergence
from torch.distributions.normal import Normal

LATENT_SCALE = 0.18215


# =============================================================================
# FAIRSEQ / ECG-FM LOADING
# =============================================================================

def _setup_fairseq_path() -> None:
    """Best-effort addition of fairseq-signals to sys.path."""
    try:
        if '__file__' in globals():
            project_root = Path(__file__).resolve().parent
            candidates = [
                project_root / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent.parent.parent / "ecg_fm_integration" / "fairseq-signals",
                project_root.parent.parent.parent.parent.parent / "ecg_fm_integration" / "fairseq-signals",
            ]
        else:
            # Notebook context or fallback
            candidates = [
                Path("/home/mithunmanivannan/ecg_fm_integration/fairseq-signals"),
                Path("/home/mithunmanivannan/third_party/WearECG-reconstruction/ecg_fm_integration/fairseq-signals"),
            ]
            
        for fs_path in candidates:
            if fs_path.exists():
                fs_path_str = str(fs_path)
                if fs_path_str not in sys.path:
                    sys.path.insert(0, fs_path_str)
                break
    except Exception:
        pass


_setup_fairseq_path()


class ECGFMFeatureExtractor(nn.Module):
    """
    Frozen ECG-FM feature extractor.

    Returns token embeddings of shape [B, T_enc, D].
    Also provides simple pooled summaries for perceptual supervision / conditioning.
    """

    def __init__(
        self,
        checkpoint_path: str,
        embed_dim: int = 768,
        allow_unexpected_keys: Optional[List[str]] = None,
    ) -> None:
        super().__init__()
        self.embed_dim = int(embed_dim)
        self.allow_unexpected_keys = set(allow_unexpected_keys or [])

        from fairseq_signals.models.ecg_transformer import ECGTransformerModel
        from fairseq_signals.utils.checkpoint_utils import load_checkpoint_to_cpu
        from omegaconf import OmegaConf

        state = load_checkpoint_to_cpu(checkpoint_path)
        cfg = state["cfg"]["model"]
        OmegaConf.set_struct(cfg, False)
        if getattr(cfg, "saliency", None) is None:
            cfg.saliency = False

        self.backbone = ECGTransformerModel.build_model(cfg)
        incompatible = self.backbone.load_state_dict(state["model"], strict=False)

        if incompatible.missing_keys:
            raise RuntimeError(f"ECG-FM checkpoint missing keys: {incompatible.missing_keys}")

        unexpected = set(incompatible.unexpected_keys)
        disallowed = sorted(unexpected - self.allow_unexpected_keys)
        if disallowed:
            raise RuntimeError(f"ECG-FM checkpoint has unsupported unexpected keys: {disallowed}")

        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()

        self.token_norm = nn.LayerNorm(self.embed_dim)

    def train(self, mode: bool = True):
        super().train(mode)
        # The FM is a frozen teacher/conditioner and should never leave eval mode.
        self.backbone.eval()
        return self

    def extract_tokens(self, x_12: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x_12: [B, 12, T], float tensor in ECG units

        Returns:
            tokens: [B, T_enc, D]
        """
        self.backbone.eval()
        x_norm = self._full12_zscore(x_12).to(torch.float32)
        device_type = x_norm.device.type
        with torch.amp.autocast(device_type, enabled=False):
            res = self.backbone.extract_features(x_norm, None)
            tokens = res["x"] if isinstance(res, dict) else res
            tokens = self.token_norm(tokens)
        return tokens.to(x_12.dtype)

    @staticmethod
    def _full12_zscore(x_12: torch.Tensor) -> torch.Tensor:
        """
        Per-lead z-score over time for fully observed 12-lead ECG.
        """
        mean = x_12.mean(dim=2, keepdim=True)
        std = x_12.std(dim=2, keepdim=True).clamp(min=1e-6)
        return (x_12 - mean) / std

    @staticmethod
    def pooled_summary(tokens: torch.Tensor) -> torch.Tensor:
        """
        Global pooled summary: [B, D]
        """
        return tokens.mean(dim=1)


# =============================================================================
# EXACT / BASELINE WEARECG BLOCKS
# =============================================================================

class SelfAttention(nn.Module):
    def __init__(self, n_heads: int, d_embed: int, in_proj_bias: bool = True, out_proj_bias: bool = True):
        super().__init__()
        self.in_proj = nn.Linear(d_embed, 3 * d_embed, bias=in_proj_bias)
        self.out_proj = nn.Linear(d_embed, d_embed, bias=out_proj_bias)
        self.n_heads = n_heads
        self.d_head = d_embed // n_heads

    def forward(self, x: torch.Tensor, causal_mask: bool = False) -> torch.Tensor:
        input_shape = x.shape
        batch_size, sequence_length, _d_embed = input_shape
        interim_shape = (batch_size, sequence_length, self.n_heads, self.d_head)

        q, k, v = self.in_proj(x).chunk(3, dim=-1)

        q = q.view(interim_shape).transpose(1, 2)
        k = k.view(interim_shape).transpose(1, 2)
        v = v.view(interim_shape).transpose(1, 2)

        weight = q @ k.transpose(-1, -2)

        if causal_mask:
            mask = torch.ones_like(weight, dtype=torch.bool).triu(1)
            weight.masked_fill_(mask, -torch.inf)

        weight = weight / math.sqrt(self.d_head)
        weight = F.softmax(weight, dim=-1)

        output = weight @ v
        output = output.transpose(1, 2)
        output = output.reshape(input_shape)
        output = self.out_proj(output)
        return output


class VAE_AttentionBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.groupnorm = nn.GroupNorm(32, channels)
        self.attention = SelfAttention(1, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x
        x = x.transpose(-1, -2)
        x = self.attention(x)
        x = x.transpose(-1, -2)
        return x + residue


class VAE_ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.groupnorm_1 = nn.GroupNorm(32, in_channels)
        self.conv_1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)

        self.groupnorm_2 = nn.GroupNorm(32, out_channels)
        self.conv_2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)

        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            self.residual_layer = nn.Conv1d(in_channels, out_channels, kernel_size=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residue = x

        x = self.groupnorm_1(x)
        x = F.silu(x)
        x = self.conv_1(x)

        x = self.groupnorm_2(x)
        x = F.silu(x)
        x = self.conv_2(x)

        return x + self.residual_layer(residue)


class VAE_Encoder(nn.Module):
    """
    Near-verbatim WearECG encoder.
    Expects x shaped [B, 12, T] or [B, T, 12] if called through wrapper.
    """

    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(12, 128, kernel_size=3, padding=1),
                VAE_ResidualBlock(128, 128),
                VAE_ResidualBlock(128, 128),
                nn.Conv1d(128, 128, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(128, 256),
                VAE_ResidualBlock(256, 256),
                nn.Conv1d(256, 256, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(256, 512),
                VAE_ResidualBlock(512, 512),
                nn.Conv1d(512, 512, kernel_size=3, stride=2, padding=0),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_ResidualBlock(512, 512),
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),
                nn.GroupNorm(32, 512),
                nn.SiLU(),
                nn.Conv1d(512, 8, kernel_size=3, padding=1),
                nn.Conv1d(8, 8, kernel_size=1, padding=0),
            ]
        )

    def forward(self, x: torch.Tensor, noise: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # If caller passed [B, T, 12], convert to [B, 12, T]
        if x.dim() != 3:
            raise ValueError(f"Expected rank-3 input, got shape {tuple(x.shape)}")
        if x.shape[1] != 12 and x.shape[2] == 12:
            x = x.transpose(1, 2)
        elif x.shape[1] != 12:
            raise ValueError(f"Expected 12 channels, got shape {tuple(x.shape)}")

        for module in self.blocks:
            if getattr(module, "stride", None) == (2,):
                x = F.pad(x, (0, 1))
            x = module(x)

        mean, log_variance = torch.chunk(x, 2, dim=1)
        log_variance = torch.clamp(log_variance, -30, 20)
        variance = log_variance.exp()
        stdev = variance.sqrt()

        if noise is None:
            noise = torch.randn_like(stdev)
        z = mean + stdev * noise
        z = z * LATENT_SCALE

        return z, mean, log_variance


class ConditionalVAE_Decoder(nn.Module):
    """
    WearECG decoder geometry with optional pooled-FM FiLM conditioning.

    Conditioning is deliberately low-risk:
    - pooled ECG-FM summary only
    - applied after residual blocks
    - zero-initialized modulation so decoder starts as baseline
    """

    def __init__(self, cond_dim: int = 0):
        super().__init__()
        self.cond_dim = int(cond_dim)

        self.blocks = nn.ModuleList(
            [
                nn.Conv1d(4, 4, kernel_size=1, padding=0),
                nn.Conv1d(4, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),   # 0
                VAE_AttentionBlock(512),
                VAE_ResidualBlock(512, 512),   # 1
                VAE_ResidualBlock(512, 512),   # 2
                VAE_ResidualBlock(512, 512),   # 3
                VAE_ResidualBlock(512, 512),   # 4
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),   # 5
                VAE_ResidualBlock(512, 512),   # 6
                VAE_ResidualBlock(512, 512),   # 7
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 512),   # 8
                VAE_ResidualBlock(512, 512),   # 9
                VAE_ResidualBlock(512, 512),   # 10
                nn.Upsample(scale_factor=2),
                nn.Conv1d(512, 512, kernel_size=3, padding=1),
                VAE_ResidualBlock(512, 256),   # 11
                VAE_ResidualBlock(256, 256),   # 12
                VAE_ResidualBlock(256, 256),   # 13
                nn.GroupNorm(32, 256),
                nn.SiLU(),
                nn.Conv1d(256, 12, kernel_size=3, padding=1),
            ]
        )

        residual_out_channels = [
            512, 512, 512, 512, 512,
            512, 512, 512,
            512, 512, 512,
            256, 256, 256,
        ]

        if self.cond_dim > 0:
            self.cond_to_film = nn.ModuleList(
                [nn.Linear(self.cond_dim, 2 * c) for c in residual_out_channels]
            )
            for layer in self.cond_to_film:
                nn.init.zeros_(layer.weight)
                nn.init.zeros_(layer.bias)
        else:
            self.cond_to_film = nn.ModuleList()

    def _apply_film(self, x: torch.Tensor, cond_vec: Optional[torch.Tensor], film_idx: int) -> torch.Tensor:
        if cond_vec is None or self.cond_dim == 0:
            return x
        gamma_beta = self.cond_to_film[film_idx](cond_vec)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        gamma = gamma.unsqueeze(-1)
        beta = beta.unsqueeze(-1)
        return x * (1.0 + gamma) + beta

    def forward(self, z: torch.Tensor, cond_vec: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = z / LATENT_SCALE
        film_idx = 0

        for module in self.blocks:
            x = module(x)
            if isinstance(module, VAE_ResidualBlock):
                x = self._apply_film(x, cond_vec, film_idx)
                film_idx += 1

        x = x.transpose(1, 2)
        return x


# =============================================================================
# LOSSES
# =============================================================================

def baseline_loss_function(
    recons: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    kld_weight: float = 1e-4,
    lead_indices: Optional[torch.Tensor] = None,
    missing_lead_weight: float = 1.0,
) -> Dict[str, torch.Tensor]:
    recons_loss = weighted_reconstruction_mse(
        recons,
        x,
        lead_indices=lead_indices,
        missing_lead_weight=missing_lead_weight,
    )

    q_z_x = Normal(mu, log_var.mul(0.5).exp())
    p_z = Normal(torch.zeros_like(mu), torch.ones_like(log_var))
    kld_loss = kl_divergence(q_z_x, p_z).sum(1).mean()

    loss = recons_loss + kld_weight * kld_loss

    return {
        "loss": loss,
        "recons_loss": recons_loss.detach(),
        "KLD_loss": kld_loss.detach(),
    }


def weighted_reconstruction_mse(
    recons: torch.Tensor,
    target: torch.Tensor,
    *,
    lead_indices: Optional[torch.Tensor] = None,
    missing_lead_weight: float = 1.0,
) -> torch.Tensor:
    """Weighted full-12 reconstruction MSE.

    Defaults to the exact baseline behavior when ``missing_lead_weight == 1.0``.
    When increased, only missing leads receive extra weight, while observed leads
    remain weight 1.0. This keeps the baseline comparison path intact while
    allowing a sparse-reconstruction emphasis ablation.
    """
    if lead_indices is None or float(missing_lead_weight) == 1.0:
        return F.mse_loss(recons, target, reduction="mean")

    if lead_indices.dim() != 2:
        raise ValueError(f"Expected lead_indices shape [B, N_obs], got {tuple(lead_indices.shape)}")
    if recons.dim() != 3 or target.dim() != 3:
        raise ValueError("Expected reconstruction and target tensors shaped [B, T, 12].")
    if recons.shape != target.shape:
        raise ValueError(f"Expected matching reconstruction/target shapes, got {tuple(recons.shape)} vs {tuple(target.shape)}")

    batch_size, _, num_leads = recons.shape
    weights = recons.new_full((batch_size, num_leads), float(missing_lead_weight))
    observed_weight = recons.new_ones((batch_size, lead_indices.shape[1]))
    weights.scatter_(1, lead_indices.long(), observed_weight)
    weights = weights.unsqueeze(1)
    squared_error = (recons - target) ** 2
    time_len = recons.shape[1]
    return (squared_error * weights).sum() / (weights.sum().clamp(min=1.0) * time_len)


def pooled_fm_perceptual_loss(
    fm_model: ECGFMFeatureExtractor,
    x_recon_12: torch.Tensor,
    pooled_true: torch.Tensor,
    *,
    cosine_mix: float = 0.5,
) -> torch.Tensor:
    """
    Pooled FM perceptual loss:
    - MSE in pooled ECG-FM space
    - optional cosine component

    Args:
        x_recon_12: [B, 12, T]
    """
    recon_tokens = fm_model.extract_tokens(x_recon_12)
    recon_pool = fm_model.pooled_summary(recon_tokens)

    mse_term = F.mse_loss(recon_pool, pooled_true, reduction="mean")
    cos_term = 1.0 - F.cosine_similarity(recon_pool, pooled_true, dim=1).mean()

    return (1.0 - cosine_mix) * mse_term + cosine_mix * cos_term


# =============================================================================
# EXACT BASELINE WRAPPER
# =============================================================================

class WearECGVAE(nn.Module):
    """
    Exact-ish baseline compatibility wrapper.

    External API:
    - stage1_forward
    - impute_from_regressor
    - forward(..., mode='stage1')
    """

    def __init__(
        self,
        in_channels: int = 12,
        out_channels: int = 12,
        latent_channels: int = 4,
        target_len: int = 5000,
        beta_kl: float = 1e-4,
        chest_weighted: bool = False,
        missing_lead_weight: float = 1.0,
    ):
        super().__init__()
        if in_channels != 12 or out_channels != 12:
            raise ValueError("WearECGVAE expects 12-lead input/output.")
        if latent_channels != 4:
            raise ValueError("Exact WearECG VAE baseline uses latent_channels=4.")

        self.target_len = int(target_len)
        self.beta_kl = float(beta_kl)
        self.chest_weighted = bool(chest_weighted)
        self.missing_lead_weight = float(missing_lead_weight)

        self.encoder = VAE_Encoder()
        self.decoder = ConditionalVAE_Decoder(cond_dim=0)

    def _match_target_len(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] < self.target_len:
            x = F.pad(x, (0, 0, 0, self.target_len - x.shape[1]))
        elif x.shape[1] > self.target_len:
            x = x[:, : self.target_len, :]
        return x

    def stage1_forward(
        self,
        x: torch.Tensor,
        y_full: Optional[torch.Tensor] = None,
        lead_indices=None,
    ) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")
        if y_full is not None and (y_full.dim() != 3 or y_full.shape[1] != 12):
            raise ValueError(f"Expected y_full shape [B, 12, T], got {tuple(y_full.shape)}")

        target_12 = y_full if y_full is not None else x
        z, mean, log_var = self.encoder(x)
        recons = self._match_target_len(self.decoder(z, cond_vec=None))
        y_pred = recons.transpose(1, 2)
        loss = baseline_loss_function(
            recons,
            target_12.transpose(1, 2),
            mean,
            log_var,
            kld_weight=self.beta_kl,
            lead_indices=lead_indices,
            missing_lead_weight=self.missing_lead_weight,
        )

        zero = torch.tensor(0.0, device=x.device)
        return {
            "loss": loss["loss"],
            "decoder_loss": loss["recons_loss"],
            "teacher_loss": zero,
            "align_loss": zero,
            "stft_loss": zero,
            "diff_loss": zero,
            "corr_loss": zero,
            "kl_loss": loss["KLD_loss"],
            "y_target": target_12,
            "y_pred": y_pred,
            "y_pred_reg": y_pred,
            "z_regressed": mean.detach(),
        }

    @torch.no_grad()
    def impute_from_regressor(self, x: torch.Tensor, lead_indices=None) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")

        _z, mean, _log_var = self.encoder(x)
        recons = self._match_target_len(self.decoder(mean * LATENT_SCALE, cond_vec=None))
        return {
            "available": True,
            "y_pred": recons.transpose(1, 2),
            "z_latent": mean.detach(),
        }

    def forward(self, x: torch.Tensor, lead_indices=None, mode: str = "stage1", **kwargs):
        if mode != "stage1":
            raise ValueError("WearECGVAE supports only mode='stage1'.")
        return self.stage1_forward(x, lead_indices=lead_indices, y_full=kwargs.get("y_full"))


# =============================================================================
# FM-INTEGRATED WRAPPER
# =============================================================================

class WearECGFMVAE(nn.Module):
    """
    WearECG baseline + ECG-FM integration.

    Integration strategy:
    1. Preserve the native WearECG encoder-decoder path.
    2. Add frozen ECG-FM perceptual supervision on reconstructions.
    3. Optionally add pooled ECG-FM decoder conditioning.

    External API matches the baseline wrapper.
    """

    def __init__(
        self,
        fm_checkpoint_path: str,
        in_channels: int = 12,
        out_channels: int = 12,
        latent_channels: int = 4,
        target_len: int = 5000,
        beta_kl: float = 1e-4,
        chest_weighted: bool = False,
        missing_lead_weight: float = 1.0,
        fm_embed_dim: int = 768,
        fm_loss_weight: float = 1e-2,
        fm_cosine_mix: float = 0.5,
        use_decoder_conditioning: bool = False,
        fm_cond_drop_prob: float = 0.0,
        use_latent_alignment: bool = False,
        latent_align_weight: float = 1e-3,
    ):
        super().__init__()
        if in_channels != 12 or out_channels != 12:
            raise ValueError("WearECGFMVAE expects 12-lead input/output.")
        if latent_channels != 4:
            raise ValueError("This implementation preserves the exact WearECG latent_channels=4 baseline.")

        self.target_len = int(target_len)
        self.beta_kl = float(beta_kl)
        self.chest_weighted = bool(chest_weighted)
        self.missing_lead_weight = float(missing_lead_weight)

        self.fm_loss_weight = float(fm_loss_weight)
        self.fm_cosine_mix = float(fm_cosine_mix)
        self.use_decoder_conditioning = bool(use_decoder_conditioning)
        self.fm_cond_drop_prob = float(fm_cond_drop_prob)
        self.use_latent_alignment = bool(use_latent_alignment)
        self.latent_align_weight = float(latent_align_weight)
        self.fm_embed_dim = int(fm_embed_dim)

        self.encoder = VAE_Encoder()

        self.fm_model = ECGFMFeatureExtractor(
            checkpoint_path=fm_checkpoint_path,
            embed_dim=fm_embed_dim,
            allow_unexpected_keys=[
                "quantizer.vars",
                "quantizer.weight_proj.weight",
                "quantizer.weight_proj.bias",
                "project_q.weight",
                "project_q.bias",
                "final_proj.weight",
                "final_proj.bias",
            ],
        )

        cond_dim = fm_embed_dim if self.use_decoder_conditioning else 0
        self.decoder = ConditionalVAE_Decoder(cond_dim=cond_dim)
        self.latent_projector = nn.Sequential(
            nn.Linear(4, 256),
            nn.SiLU(),
            nn.Linear(256, self.fm_embed_dim),
        )

    def _match_target_len(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] < self.target_len:
            x = F.pad(x, (0, 0, 0, self.target_len - x.shape[1]))
        elif x.shape[1] > self.target_len:
            x = x[:, : self.target_len, :]
        return x

    def _get_decoder_condition(self, x_12: torch.Tensor) -> Optional[torch.Tensor]:
        if not self.use_decoder_conditioning:
            return None

        with torch.no_grad():
            tokens = self.fm_model.extract_tokens(x_12)
            cond_vec = self.fm_model.pooled_summary(tokens)

        if self.training and self.fm_cond_drop_prob > 0.0:
            keep = (torch.rand(cond_vec.size(0), device=cond_vec.device) > self.fm_cond_drop_prob).float()
            cond_vec = cond_vec * keep.unsqueeze(1)

        return cond_vec.to(x_12.dtype)

    def _latent_align_loss(self, mean: torch.Tensor, pooled_true: torch.Tensor) -> torch.Tensor:
        if not self.use_latent_alignment:
            return mean.new_tensor(0.0)
        mean_pool = mean.float().mean(dim=2)
        pred = self.latent_projector(mean_pool)
        return F.mse_loss(pred, pooled_true.detach().float(), reduction="mean")

    def stage1_forward(
        self,
        x: torch.Tensor,
        y_full: Optional[torch.Tensor] = None,
        lead_indices=None,
    ) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")
        if y_full is not None and (y_full.dim() != 3 or y_full.shape[1] != 12):
            raise ValueError(f"Expected y_full shape [B, 12, T], got {tuple(y_full.shape)}")

        target_12 = y_full if y_full is not None else x
        # Baseline path
        z, mean, log_var = self.encoder(x)

        cond_vec = self._get_decoder_condition(x)
        recons = self._match_target_len(self.decoder(z, cond_vec=cond_vec))
        y_pred = recons.transpose(1, 2)

        base_loss = baseline_loss_function(
            recons,
            target_12.transpose(1, 2),
            mean,
            log_var,
            kld_weight=self.beta_kl,
            lead_indices=lead_indices,
            missing_lead_weight=self.missing_lead_weight,
        )

        # FM perceptual supervision
        with torch.no_grad():
            true_tokens = self.fm_model.extract_tokens(target_12)
            pooled_true = self.fm_model.pooled_summary(true_tokens)

        fm_teacher = pooled_fm_perceptual_loss(
            fm_model=self.fm_model,
            x_recon_12=y_pred,
            pooled_true=pooled_true.detach(),
            cosine_mix=self.fm_cosine_mix,
        )
        latent_align = self._latent_align_loss(mean, pooled_true)

        total_loss = base_loss["loss"] + self.fm_loss_weight * fm_teacher + self.latent_align_weight * latent_align
        zero = x.new_tensor(0.0)

        return {
            "loss": total_loss,
            "decoder_loss": base_loss["recons_loss"],
            "teacher_loss": fm_teacher.detach(),
            "align_loss": latent_align.detach() if self.use_latent_alignment else zero,
            "stft_loss": zero,
            "diff_loss": zero,
            "corr_loss": zero,
            "kl_loss": base_loss["KLD_loss"],
            "y_target": target_12,
            "y_pred": y_pred,
            "y_pred_reg": y_pred,
            "z_regressed": mean.detach(),
            "fm_perceptual_loss": fm_teacher.detach(),
            "latent_align_loss": latent_align.detach() if self.use_latent_alignment else zero,
        }

    @torch.no_grad()
    def impute_from_regressor(self, x: torch.Tensor, lead_indices=None) -> Dict[str, torch.Tensor]:
        if x.dim() != 3 or x.shape[1] != 12:
            raise ValueError(f"Expected x shape [B, 12, T], got {tuple(x.shape)}")

        _z, mean, _log_var = self.encoder(x)

        cond_vec = self._get_decoder_condition(x)
        recons = self._match_target_len(self.decoder(mean * LATENT_SCALE, cond_vec=cond_vec))

        return {
            "available": True,
            "y_pred": recons.transpose(1, 2),
            "z_latent": mean.detach(),
        }

    def forward(self, x: torch.Tensor, lead_indices=None, mode: str = "stage1", **kwargs):
        if mode != "stage1":
            raise ValueError("WearECGFMVAE supports only mode='stage1'.")
        return self.stage1_forward(x, lead_indices=lead_indices, y_full=kwargs.get("y_full"))


### Training Engine (FM Integration) (train_vae_fm.py)


In [15]:
"""Train the exact WearECG baseline or the FM-assisted WearECG VAE (Demo Mode)."""

from __future__ import annotations
import argparse
import math
import os
import sys
from typing import Any
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import wandb

# [STABILITY FIX] Ensure any previous run is closed
try:
    wandb.finish()
except:
    pass

sys.path.append(os.getcwd())

from src.reconstruction.unified_latents.engineering.common import (
    TensorFolderDataset,
    cleanup_partial_checkpoints,
    prune_epoch_checkpoints,
    write_best_summary,
    write_run_artifacts,
    write_warm_start_summary,
)
from src.reconstruction.unified_latents.engineering.eval_reconstruction import evaluate_reconstruction
from src.reconstruction.unified_latents.engineering.regimes import (
    LEAD_NAMES,
    format_lead_set,
    get_missing_indices,
    make_lead_indices,
    resolve_obs_leads,
)
from src.reconstruction.unified_latents.engineering.vae_fm import WearECGVAE, WearECGFMVAE

DEFAULT_FM_CKPT = "/home/mithunmanivannan/ecg_fm_integration/checkpoints/mimic_iv_ecg_physionet_pretrained.pt"

def add_bool_arg(parser: argparse.ArgumentParser, name: str, default: bool) -> None:
    dest = name.replace("-", "_")
    parser.add_argument(f"--{name}", dest=dest, action="store_true")
    parser.add_argument(f"--no-{name}", dest=dest, action="store_false")
    parser.set_defaults(**{dest: default})

def get_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(allow_abbrev=False)
    parser.add_argument("--model_family", type=str, choices=["baseline", "fm_vae"], default="fm_vae")
    parser.add_argument("--epochs", type=int, default=1)  # Set to 1 for demo
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=1e-5)
    parser.add_argument("--max_lr", type=float, default=5e-5)
    parser.add_argument("--target_len", type=int, default=5000)
    parser.add_argument("--latent_channels", type=int, default=4)
    parser.add_argument("--beta_kl", type=float, default=1e-4)
    parser.add_argument("--missing_lead_weight", type=float, default=1.0)
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--regime", type=str, choices=["current", "wearecg", "historical"], default="wearecg")
    parser.add_argument("--obs_leads", type=str, default=None)
    parser.add_argument("--run_tag", type=str, default="demo_run")
    parser.add_argument("--split", type=str, choices=["val"], default="val")
    parser.add_argument("--debug", action="store_true", default=True) # Enabled for demo
    parser.add_argument("--debug_logs", action="store_true", default=True)
    parser.add_argument("--save_training_state", action="store_true")
    parser.add_argument("--fm_checkpoint", type=str, default=DEFAULT_FM_CKPT)
    parser.add_argument("--fm_loss_weight", type=float, default=1e-2)
    parser.add_argument("--fm_cosine_mix", type=float, default=0.5)
    parser.add_argument("--fm_cond_drop_prob", type=float, default=0.0)
    parser.add_argument("--latent_align_weight", type=float, default=1e-3)
    add_bool_arg(parser, "fm_perceptual", True)
    add_bool_arg(parser, "fm_decoder_conditioning", False)
    add_bool_arg(parser, "fm_latent_align", False)
    
    # [FIX] Handle Jupyter connection file arguments
    args, _ = parser.parse_known_args()
    args.obs_lead_indices = resolve_obs_leads(args.regime, args.obs_leads)
    return args

def train(args: argparse.Namespace) -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    lead_tag = format_lead_set(args.obs_lead_indices)
    model_family_tag = "fm_vae" if args.model_family == "fm_vae" else "baseline"
    run_name = f"wearecg_fm_demo_{model_family_tag}_{lead_tag}"

    # [STABILITY FIX] Use reinit=True for notebook safety
    wandb.init(
        project="WearECG-FM-Scientific-Production",
        name=run_name,
        config=vars(args),
        reinit=True
    )

    # [FIX] Using absolute data directory
    base_dir = "/home/mithunmanivannan/data/ptb_xl/tensors"
    train_dataset = TensorFolderDataset(f"{base_dir}/train")
    if args.debug:
        train_dataset = torch.utils.data.Subset(train_dataset, range(128)) # Small subset for demo

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=4)

    # [DEMO] Outputs redirected to demo_outputs
    save_dir = f"demo_outputs/engineering_{model_family_tag}_{lead_tag}"
    os.makedirs(save_dir, exist_ok=True)

    model = WearECGFMVAE(
        fm_checkpoint_path=args.fm_checkpoint,
        latent_channels=4,
        target_len=args.target_len,
        beta_kl=args.beta_kl,
        missing_lead_weight=args.missing_lead_weight
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    
    print(f"Starting Demo Training... saving to: {save_dir}")
    for epoch in range(args.epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f"Demo Epoch {epoch + 1}")
        for x, y, _ in pbar:
            x, y = x.to(device), y.to(device)
            loss_dict = model(x, y_full=y, lead_indices=make_lead_indices(args.obs_lead_indices, x.size(0), device), mode="stage1")
            loss = loss_dict["loss"]
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            wandb.log({"train/loss": float(loss.item())})
            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

    # Save demo checkpoint
    ckpt_path = os.path.join(save_dir, "demo_final.pt")
    torch.save({"model_state_dict": model.state_dict()}, ckpt_path)
    print(f"Demo complete. Checkpoint saved to {ckpt_path}")
    wandb.finish()

if __name__ == "__main__":
    train(get_args())


socket.send() raised exception.
socket.send() raised exception.


Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x713f58d3f020>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7140351d3860, raw_cell=""""Train the exact WearECG baseline or the FM-assi.." transformed_cell=""""Train the exact WearECG baseline or the FM-assi.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a223133342e3131372e3231342e313631227d/home/mithunmanivannan/third_party/WearECG-reconstruction/VAE.ipynb#Z3033sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

socket.send() raised exception.
socket.send() raised exception.


socket.send() raised exception.
socket.send() raised exception.


BrokenPipeError: [Errno 32] Broken pipe

socket.send() raised exception.
socket.send() raised exception.


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x713f58d3f020>> (for post_run_cell), with arguments args (<ExecutionResult object at 7140351d1d00, execution_count=15 error_before_exec=None error_in_exec=[Errno 32] Broken pipe info=<ExecutionInfo object at 7140351d3860, raw_cell=""""Train the exact WearECG baseline or the FM-assi.." transformed_cell=""""Train the exact WearECG baseline or the FM-assi.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a223133342e3131372e3231342e313631227d/home/mithunmanivannan/third_party/WearECG-reconstruction/VAE.ipynb#Z3033sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

### Evaluation Script (eval_reconstruction.py)


In [ ]:
"""Unified evaluator for engineering reconstruction checkpoints."""

from __future__ import annotations

import argparse
import contextlib
import os
import sys

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.append(os.getcwd())

from src.reconstruction.evaluate_functions.fiducials import FS_HZ, compute_reconstruction_morphology_metrics
from src.reconstruction.unified_latents.engineering.common import (
    CHEST_INDICES_ALL,
    CHEST_LEADS,
    LATERAL_LEADS,
    RECON_FOCUS_LEADS,
    TensorFolderDataset,
    V3_V6_LEADS,
    V4_V6_LEADS,
    compute_batch_corr_per_lead,
    compute_batch_mae,
    compute_batch_mae_per_lead,
    compute_batch_mse,
    compute_batch_mse_per_lead,
    compute_batch_r2,
    compute_batch_r2_per_lead,
    compute_batch_rmse,
    compute_batch_rmse_per_lead,
    compute_rwave_progression_metrics,
    mean_for_leads,
    to_serializable_metrics,
    write_json,
    write_run_artifacts,
)
from src.reconstruction.unified_latents.engineering.regimes import (
    LEAD_NAMES,
    format_lead_set,
    get_missing_indices,
    make_lead_indices,
    resolve_obs_leads,
)


def _autocast_context(device):
    if device.type != "cuda":
        return contextlib.nullcontext()
    return torch.amp.autocast("cuda", dtype=torch.bfloat16)


def _checkpoint_state_dict(path: str):
    ckpt = torch.load(path, map_location="cpu")
    return ckpt.get("model_state_dict", ckpt)


def _checkpoint_payload(path: str):
    return torch.load(path, map_location="cpu")


def _named_metric(metrics: dict[str, object], key: str) -> float:
    value = metrics.get(key, float("nan"))
    if isinstance(value, torch.Tensor):
        return float(value.item())
    return float(value)


def _load_checkpoint_state_strict(model: torch.nn.Module, state_dict: dict[str, object], model_family: str) -> None:
    model_state = model.state_dict()
    model_keys = set(model_state.keys())
    if model_family == "fm_vae":
        excluded_prefixes = ("fm_model.backbone.",)
        required_model_keys = {key for key in model_keys if not key.startswith(excluded_prefixes)}
    else:
        excluded_prefixes = ()
        required_model_keys = model_keys

    provided_keys = set(state_dict.keys())
    missing_keys = sorted(required_model_keys - provided_keys)
    unexpected_keys = sorted(provided_keys - model_keys)
    shape_mismatches = sorted(
        key for key in (required_model_keys & provided_keys) if model_state[key].shape != state_dict[key].shape
    )
    if missing_keys or unexpected_keys or shape_mismatches:
        raise RuntimeError(
            "Checkpoint/model mismatch. "
            f"missing={missing_keys[:12]} unexpected={unexpected_keys[:12]} "
            f"shape_mismatches={shape_mismatches[:12]}"
        )

    incompatible = model.load_state_dict(state_dict, strict=False)
    remaining_missing = [
        key for key in getattr(incompatible, "missing_keys", [])
        if not key.startswith(excluded_prefixes)
    ]
    remaining_unexpected = list(getattr(incompatible, "unexpected_keys", []))
    if remaining_missing or remaining_unexpected:
        raise RuntimeError(
            "Unexpected incompatibility after guarded checkpoint load. "
            f"missing={remaining_missing[:12]} unexpected={remaining_unexpected[:12]}"
        )


def _print_eval_summary(
    metrics: dict[str, object],
    *,
    split: str,
    model_family: str,
    obs_indices: list[int],
    debug: bool = False,
) -> None:
    learned_target_indices = get_missing_indices(obs_indices)
    learned_target_names = [LEAD_NAMES[idx] for idx in learned_target_indices]
    print(f"\n[EvalSummary] split={split} model_family={model_family}")
    print(f"  observed={ [LEAD_NAMES[idx] for idx in obs_indices] }")
    print(f"  learned_targets={learned_target_names}")
    print(
        "  core "
        f"r2_reg={_named_metric(metrics, f'{split}/r2_regressor'):.4f} "
        f"mse={_named_metric(metrics, f'{split}/mse_reg'):.6f} "
        f"mae={_named_metric(metrics, f'{split}/mae_reg'):.6f} "
        f"rmse={_named_metric(metrics, f'{split}/rmse_reg'):.6f}"
    )
    print(
        "  losses "
        f"decoder={_named_metric(metrics, f'{split}/decoder_loss'):.6f} "
        f"kl={_named_metric(metrics, f'{split}/kl_loss'):.6f} "
        f"fm={_named_metric(metrics, f'{split}/fm_perceptual_loss'):.6f} "
        f"latent_align={_named_metric(metrics, f'{split}/latent_align_loss'):.6f}"
    )
    if f"{split}/r2_teacher_clean" in metrics:
        print(
            "  teacher "
            f"r2={_named_metric(metrics, f'{split}/r2_teacher_clean'):.4f} "
            f"mse_z={_named_metric(metrics, f'{split}/mse_z_reg'):.6f}"
        )
    print(
        "  aggregates "
        f"v4_v6={_named_metric(metrics, f'{split}/r2_reg_v4_v6_mean'):.4f} "
        f"v3_v6={_named_metric(metrics, f'{split}/r2_reg_v3_v6_mean'):.4f} "
        f"chest={_named_metric(metrics, f'{split}/r2_reg_chest_mean'):.4f} "
        f"lateral={_named_metric(metrics, f'{split}/r2_reg_lateral_mean'):.4f}"
    )
    if f"{split}/clinical_reg_recon_chest_mean_beat_corr" in metrics:
        print(
            "  morphology "
            f"beat_corr={_named_metric(metrics, f'{split}/clinical_reg_recon_chest_mean_beat_corr'):.6f} "
            f"beat_rmse={_named_metric(metrics, f'{split}/clinical_reg_recon_chest_mean_beat_rmse'):.6f} "
            f"r_peak_ms={_named_metric(metrics, f'{split}/clinical_reg_recon_chest_mean_r_peak_timing_error_ms'):.6f}"
        )
    if not debug:
        return

    for lead_name in learned_target_names:
        print(
            "    "
            f"{lead_name} | "
            f"r2={_named_metric(metrics, f'{split}/lead_r2_reg_{lead_name}'):.4f} "
            f"mae={_named_metric(metrics, f'{split}/mae_reg_{lead_name}'):.5f} "
            f"mse={_named_metric(metrics, f'{split}/mse_reg_{lead_name}'):.5f} "
            f"rmse={_named_metric(metrics, f'{split}/rmse_reg_{lead_name}'):.5f} "
            f"corr={_named_metric(metrics, f'{split}/corr_reg_{lead_name}'):.4f}"
        )
    focus_keys = [
        f"{split}/clinical_reg_recon_V4_mean_beat_corr",
        f"{split}/clinical_reg_recon_V4_mean_beat_rmse",
        f"{split}/clinical_reg_recon_V4_mean_r_peak_timing_error_ms",
        f"{split}/clinical_reg_recon_V5_mean_beat_corr",
        f"{split}/clinical_reg_recon_V5_mean_beat_rmse",
        f"{split}/clinical_reg_recon_V5_mean_r_peak_timing_error_ms",
        f"{split}/clinical_reg_recon_V6_mean_beat_corr",
        f"{split}/clinical_reg_recon_V6_mean_beat_rmse",
        f"{split}/clinical_reg_recon_V6_mean_r_peak_timing_error_ms",
    ]
    present_focus = [key for key in focus_keys if key in metrics]
    if present_focus:
        print("  focus_morphology")
        for key in present_focus:
            print(f"    {key.split('/', 1)[1]}={_named_metric(metrics, key):.6f}")


def evaluate_reconstruction(
    model,
    val_loader,
    device,
    obs_indices: list[int],
    *,
    split: str = "val",
    step: int | None = None,
    model_family: str = "hybrid",
    log_to_wandb: bool = False,
) -> dict[str, float]:
    import wandb

    was_training = model.training
    model.eval()
    gen_target_indices = get_missing_indices(obs_indices)
    lead_names = [LEAD_NAMES[idx] for idx in gen_target_indices]

    recon_chest_indices = [idx for idx in gen_target_indices if idx in [8, 9, 10, 11]]
    if not recon_chest_indices:
        recon_chest_indices = [idx for idx in gen_target_indices if idx in CHEST_INDICES_ALL]

    teacher_available = model_family == "hybrid" and hasattr(model, "impute_from_teacher")
    teacher_r2_total = 0.0
    teacher_r2_leads = [0.0] * len(gen_target_indices)
    reg_r2_total = 0.0
    reg_r2_leads = [0.0] * len(gen_target_indices)
    reg_mae_leads = [0.0] * len(gen_target_indices)
    reg_mse_leads = [0.0] * len(gen_target_indices)
    reg_rmse_leads = [0.0] * len(gen_target_indices)
    reg_corr_leads = [0.0] * len(gen_target_indices)

    val_loss_decoder = 0.0
    val_loss_teacher = 0.0
    val_loss_align = 0.0
    val_loss_stft = 0.0
    val_loss_diff = 0.0
    val_loss_corr = 0.0
    val_loss_kl = 0.0
    val_loss_fm_perceptual = 0.0
    val_loss_latent_align = 0.0

    mse_z_reg_total = 0.0
    reg_mae_total = 0.0
    reg_mse_total = 0.0
    reg_rmse_total = 0.0
    reg_batches = 0
    latent_batches = 0

    reg_morph_total = {
        "samples_with_beats": 0.0,
        "mean_beat_rmse": 0.0,
        "mean_beat_corr": 0.0,
        "mean_r_amp_error": 0.0,
        "mean_r_peak_timing_error_ms": 0.0,
    }
    reg_morph_batches = 0
    reg_progression_total = {"rwave_progression_mae": 0.0, "rwave_progression_corr": 0.0}
    reg_progression_batches = 0
    reg_progression_recon_total = {"rwave_progression_mae": 0.0, "rwave_progression_corr": 0.0}
    reg_progression_recon_batches = 0
    reg_morph_focus_total = {
        lead_name: {
            "mean_beat_rmse": 0.0,
            "mean_beat_corr": 0.0,
            "mean_r_peak_timing_error_ms": 0.0,
        }
        for lead_name, lead_idx in RECON_FOCUS_LEADS
        if lead_idx in gen_target_indices
    }
    reg_morph_focus_batches = {lead_name: 0 for lead_name in reg_morph_focus_total}

    with torch.no_grad():
        for x, y, _ in tqdm(val_loader, desc=f"Evaluate-{split}"):
            x = x.to(device, dtype=torch.float32, non_blocking=True)
            y = y.to(device, dtype=torch.float32, non_blocking=True)
            lead_indices = make_lead_indices(obs_indices, x.size(0), device)

            with _autocast_context(device):
                out = model(x, y_full=y, lead_indices=lead_indices, mode="stage1")
                reg_out = model.impute_from_regressor(x, lead_indices=lead_indices)
                teacher_out = model.impute_from_teacher(x, lead_indices=lead_indices) if teacher_available else None

            if not reg_out.get("available", True):
                raise RuntimeError("Primary reconstruction path is unavailable during evaluation.")

            y_target_miss = y[:, gen_target_indices, :]
            y_pred_reg = reg_out["y_pred"]
            reg_pred_miss = y_pred_reg[:, gen_target_indices, :]
            z_reg = reg_out.get("z_latent")

            teacher_pred_miss = None
            z_teacher = None
            if teacher_out is not None:
                y_pred_teacher = teacher_out["y_pred"]
                teacher_pred_miss = y_pred_teacher[:, gen_target_indices, :]
                z_teacher = teacher_out.get("z_clean")

            # Evaluation uses autocast for speed, but downstream morphology helpers
            # expect float tensors when they leave PyTorch for NumPy-based fiducials.
            y_float = y.float()
            y_pred_reg_float = y_pred_reg.float()

            val_loss_decoder += float(out.get("decoder_loss", torch.tensor(0.0, device=device)).item())
            val_loss_teacher += float(out.get("teacher_loss", torch.tensor(0.0, device=device)).item())
            val_loss_align += float(out.get("align_loss", torch.tensor(0.0, device=device)).item())
            val_loss_stft += float(out.get("stft_loss", torch.tensor(0.0, device=device)).item())
            val_loss_diff += float(out.get("diff_loss", torch.tensor(0.0, device=device)).item())
            val_loss_corr += float(out.get("corr_loss", torch.tensor(0.0, device=device)).item())
            val_loss_kl += float(out.get("kl_loss", torch.tensor(0.0, device=device)).item())
            val_loss_fm_perceptual += float(out.get("fm_perceptual_loss", out.get("teacher_loss", torch.tensor(0.0, device=device))).item())
            val_loss_latent_align += float(out.get("latent_align_loss", out.get("align_loss", torch.tensor(0.0, device=device))).item())

            reg_r2_total += compute_batch_r2(reg_pred_miss, y_target_miss).item()
            reg_mae_total += compute_batch_mae(reg_pred_miss, y_target_miss).item()
            reg_mse_total += compute_batch_mse(reg_pred_miss, y_target_miss).item()
            reg_rmse_total += compute_batch_rmse(reg_pred_miss, y_target_miss).item()
            reg_batches += 1

            if teacher_pred_miss is not None:
                teacher_r2_total += compute_batch_r2(teacher_pred_miss, y_target_miss).item()
            if z_teacher is not None and z_reg is not None:
                mse_z_reg_total += torch.nn.functional.mse_loss(z_reg, z_teacher).item()
                latent_batches += 1

            teacher_lead_vals = compute_batch_r2_per_lead(teacher_pred_miss, y_target_miss) if teacher_pred_miss is not None else None
            reg_lead_vals = compute_batch_r2_per_lead(reg_pred_miss, y_target_miss)
            reg_mae_vals = compute_batch_mae_per_lead(reg_pred_miss, y_target_miss)
            reg_mse_vals = compute_batch_mse_per_lead(reg_pred_miss, y_target_miss)
            reg_rmse_vals = compute_batch_rmse_per_lead(reg_pred_miss, y_target_miss)
            reg_corr_vals = compute_batch_corr_per_lead(reg_pred_miss, y_target_miss)
            morph_metrics = compute_reconstruction_morphology_metrics(y_float, y_pred_reg_float, lead_indices=recon_chest_indices, fs=FS_HZ)
            progression_metrics_full = compute_rwave_progression_metrics(y_float, y_pred_reg_float, chest_indices=CHEST_INDICES_ALL)
            progression_metrics_recon = compute_rwave_progression_metrics(y_float, y_pred_reg_float, chest_indices=recon_chest_indices)

            for i in range(len(gen_target_indices)):
                if teacher_lead_vals is not None:
                    teacher_r2_leads[i] += teacher_lead_vals[i]
                reg_r2_leads[i] += reg_lead_vals[i]
                reg_mae_leads[i] += reg_mae_vals[i]
                reg_mse_leads[i] += reg_mse_vals[i]
                reg_rmse_leads[i] += reg_rmse_vals[i]
                reg_corr_leads[i] += reg_corr_vals[i]

            if not torch.isnan(torch.tensor(morph_metrics["mean_beat_rmse"])):
                reg_morph_batches += 1
                for key in reg_morph_total:
                    reg_morph_total[key] += morph_metrics[key]
            if not torch.isnan(torch.tensor(progression_metrics_full["rwave_progression_mae"])):
                reg_progression_batches += 1
                for key in reg_progression_total:
                    reg_progression_total[key] += progression_metrics_full[key]
            if not torch.isnan(torch.tensor(progression_metrics_recon["rwave_progression_mae"])):
                reg_progression_recon_batches += 1
                for key in reg_progression_recon_total:
                    reg_progression_recon_total[key] += progression_metrics_recon[key]

            for lead_name, lead_idx in RECON_FOCUS_LEADS:
                if lead_idx not in gen_target_indices:
                    continue
                per_lead_metrics = compute_reconstruction_morphology_metrics(y_float, y_pred_reg_float, lead_indices=[lead_idx], fs=FS_HZ)
                if torch.isnan(torch.tensor(per_lead_metrics["mean_beat_rmse"])):
                    continue
                reg_morph_focus_batches[lead_name] += 1
                reg_morph_focus_total[lead_name]["mean_beat_rmse"] += per_lead_metrics["mean_beat_rmse"]
                reg_morph_focus_total[lead_name]["mean_beat_corr"] += per_lead_metrics["mean_beat_corr"]
                reg_morph_focus_total[lead_name]["mean_r_peak_timing_error_ms"] += per_lead_metrics["mean_r_peak_timing_error_ms"]

    n = max(reg_batches, 1)
    teacher_r2_total = teacher_r2_total / n if teacher_available else float("nan")
    reg_r2_total /= n
    reg_mae_total /= n
    reg_mse_total /= n
    reg_rmse_total /= n
    val_loss_decoder /= n
    val_loss_teacher /= n
    val_loss_align /= n
    val_loss_stft /= n
    val_loss_diff /= n
    val_loss_corr /= n
    val_loss_kl /= n
    val_loss_fm_perceptual /= n
    val_loss_latent_align /= n
    mse_z_reg_total = mse_z_reg_total / max(latent_batches, 1) if latent_batches > 0 else float("nan")

    for i in range(len(gen_target_indices)):
        if teacher_available:
            teacher_r2_leads[i] /= n
        reg_r2_leads[i] /= n
        reg_mae_leads[i] /= n
        reg_mse_leads[i] /= n
        reg_rmse_leads[i] /= n
        reg_corr_leads[i] /= n

    metrics = {
        f"{split}/decoder_loss": val_loss_decoder,
        f"{split}/teacher_loss": val_loss_teacher,
        f"{split}/align_loss": val_loss_align,
        f"{split}/stft_loss": val_loss_stft,
        f"{split}/diff_loss": val_loss_diff,
        f"{split}/corr_loss": val_loss_corr,
        f"{split}/kl_loss": val_loss_kl,
        f"{split}/fm_perceptual_loss": val_loss_fm_perceptual,
        f"{split}/latent_align_loss": val_loss_latent_align,
        f"{split}/r2_regressor": reg_r2_total,
        f"{split}/mae_reg": reg_mae_total,
        f"{split}/mse_reg": reg_mse_total,
        f"{split}/rmse_reg": reg_rmse_total,
    }
    if teacher_available:
        metrics[f"{split}/r2_teacher_clean"] = teacher_r2_total
    if latent_batches > 0:
        metrics[f"{split}/mse_z_reg"] = mse_z_reg_total

    for i, lead in enumerate(lead_names):
        if teacher_available:
            metrics[f"{split}/lead_r2_teach_{lead}"] = teacher_r2_leads[i]
        metrics[f"{split}/lead_r2_reg_{lead}"] = reg_r2_leads[i]
        metrics[f"{split}/mae_reg_{lead}"] = reg_mae_leads[i]
        metrics[f"{split}/mse_reg_{lead}"] = reg_mse_leads[i]
        metrics[f"{split}/rmse_reg_{lead}"] = reg_rmse_leads[i]
        metrics[f"{split}/corr_reg_{lead}"] = reg_corr_leads[i]

    if teacher_available:
        teacher_chest_mean = mean_for_leads(teacher_r2_leads, lead_names, CHEST_LEADS)
        teacher_lateral_mean = mean_for_leads(teacher_r2_leads, lead_names, LATERAL_LEADS)
        if teacher_chest_mean is not None:
            metrics[f"{split}/r2_teach_chest_mean"] = teacher_chest_mean
        if teacher_lateral_mean is not None:
            metrics[f"{split}/r2_teach_lateral_mean"] = teacher_lateral_mean

    reg_chest_mean = mean_for_leads(reg_r2_leads, lead_names, CHEST_LEADS)
    reg_lateral_mean = mean_for_leads(reg_r2_leads, lead_names, LATERAL_LEADS)
    reg_v4_v6_mean = mean_for_leads(reg_r2_leads, lead_names, V4_V6_LEADS)
    reg_v3_v6_mean = mean_for_leads(reg_r2_leads, lead_names, V3_V6_LEADS)
    reg_mae_chest_mean = mean_for_leads(reg_mae_leads, lead_names, CHEST_LEADS)
    reg_rmse_chest_mean = mean_for_leads(reg_rmse_leads, lead_names, CHEST_LEADS)
    reg_corr_chest_mean = mean_for_leads(reg_corr_leads, lead_names, CHEST_LEADS)
    reg_mae_lateral_mean = mean_for_leads(reg_mae_leads, lead_names, LATERAL_LEADS)
    reg_rmse_lateral_mean = mean_for_leads(reg_rmse_leads, lead_names, LATERAL_LEADS)
    reg_corr_lateral_mean = mean_for_leads(reg_corr_leads, lead_names, LATERAL_LEADS)

    if reg_chest_mean is not None:
        metrics[f"{split}/r2_reg_chest_mean"] = reg_chest_mean
    if reg_lateral_mean is not None:
        metrics[f"{split}/r2_reg_lateral_mean"] = reg_lateral_mean
    if reg_v4_v6_mean is not None:
        metrics[f"{split}/r2_reg_v4_v6_mean"] = reg_v4_v6_mean
    if reg_v3_v6_mean is not None:
        metrics[f"{split}/r2_reg_v3_v6_mean"] = reg_v3_v6_mean
    if reg_mae_chest_mean is not None:
        metrics[f"{split}/mae_reg_chest_mean"] = reg_mae_chest_mean
        metrics[f"{split}/rmse_reg_chest_mean"] = reg_rmse_chest_mean
        metrics[f"{split}/corr_reg_chest_mean"] = reg_corr_chest_mean
    if reg_mae_lateral_mean is not None:
        metrics[f"{split}/mae_reg_lateral_mean"] = reg_mae_lateral_mean
        metrics[f"{split}/rmse_reg_lateral_mean"] = reg_rmse_lateral_mean
        metrics[f"{split}/corr_reg_lateral_mean"] = reg_corr_lateral_mean
    if teacher_available:
        metrics[f"{split}/gap_reg_vs_teacher"] = reg_r2_total - teacher_r2_total
        if reg_chest_mean is not None and f"{split}/r2_teach_chest_mean" in metrics:
            metrics[f"{split}/gap_reg_vs_teacher_chest_mean"] = reg_chest_mean - metrics[f"{split}/r2_teach_chest_mean"]
        if reg_lateral_mean is not None and f"{split}/r2_teach_lateral_mean" in metrics:
            metrics[f"{split}/gap_reg_vs_teacher_lateral_mean"] = reg_lateral_mean - metrics[f"{split}/r2_teach_lateral_mean"]

    if reg_morph_batches > 0:
        for key, total in reg_morph_total.items():
            metrics[f"{split}/clinical_reg_recon_chest_{key}"] = total / reg_morph_batches
    if reg_progression_batches > 0:
        for key, total in reg_progression_total.items():
            metrics[f"{split}/clinical_reg_full_chest_{key}"] = total / reg_progression_batches
    if reg_progression_recon_batches > 0:
        for key, total in reg_progression_recon_total.items():
            metrics[f"{split}/clinical_reg_recon_chest_{key}"] = total / reg_progression_recon_batches
    for lead_name, lead_totals in reg_morph_focus_total.items():
        batch_count = reg_morph_focus_batches[lead_name]
        if batch_count <= 0:
            continue
        for key, total in lead_totals.items():
            metrics[f"{split}/clinical_reg_recon_{lead_name}_{key}"] = total / batch_count

    if log_to_wandb and wandb.run is not None:
        wandb.log(metrics, step=step)

    if was_training:
        model.train()
    return metrics


def build_eval_metadata(args, obs_indices: list[int], payload: dict[str, object] | None = None) -> dict[str, object]:
    external_status = "gated_internal_only" if (args.include_sunnybrook or args.include_ecgfounder) else "not_requested"
    metadata = {
        "family": "engineering",
        "model_family": args.model_family,
        "regime": args.regime,
        "obs_leads": [LEAD_NAMES[idx] for idx in obs_indices],
        "obs_lead_indices": obs_indices,
        "split": args.split,
        "checkpoint": args.checkpoint,
        "include_sunnybrook": bool(args.include_sunnybrook),
        "include_ecgfounder": bool(args.include_ecgfounder),
        "external_status": external_status,
    }
    if args.model_family == "fm_vae":
        metadata.update(
            {
                "comparison_protocol": "wear_ecg_exact_regime",
                "fm_features_active": True,
                "missing_lead_weight": (payload or {}).get("missing_lead_weight", args.missing_lead_weight),
                "checkpoint_contains_fm_backbone": False,
            }
        )
    return metadata


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--model_family", type=str, choices=["hybrid", "wearecg_vae", "fm_vae"], required=True)
    parser.add_argument("--regime", type=str, choices=["current", "wearecg", "historical"], default="current")
    parser.add_argument("--obs_leads", type=str, default=None)
    parser.add_argument("--split", type=str, choices=["val", "test"], default="val")
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--target_len", type=int, default=5000)
    parser.add_argument("--latent_dim", type=int, default=32)
    parser.add_argument("--latent_channels", type=int, default=64)
    parser.add_argument("--missing_lead_weight", type=float, default=1.0)
    parser.add_argument("--output_dir", type=str, default=None)
    parser.add_argument("--fm_checkpoint", type=str, default="ecg_fm_integration/checkpoints/mimic_iv_ecg_physionet_pretrained.pt")
    parser.add_argument("--include_sunnybrook", action="store_true")
    parser.add_argument("--include_ecgfounder", action="store_true")
    parser.add_argument("--debug", action="store_true")
    return parser.parse_args()


def load_model(args, device):
    payload = _checkpoint_payload(args.checkpoint)
    state_dict = payload.get("model_state_dict", payload)
    if args.model_family == "hybrid":
        from src.reconstruction.unified_latents.engineering.ul_ecg import UL_ConditionalBridge

        fm_checkpoint = payload.get("fm_checkpoint", args.fm_checkpoint)
        model = UL_ConditionalBridge(
            checkpoint_path=fm_checkpoint,
            freeze_backbone=True,
            target_len=payload.get("target_len", args.target_len),
            teacher_loss_weight=payload.get("teacher_loss_weight", 0.10),
            reg_loss_weight=payload.get("reg_loss_weight", 1.0),
            align_loss_weight=payload.get("align_loss_weight", 0.5),
            finetune_mode=payload.get("finetune_mode", "anchored"),
            use_fm_perceptual=True,
            fm_perceptual_weight=payload.get("fm_perceptual_weight", 0.10),
        )
        model.freeze_teacher_encoder = bool(payload.get("freeze_teacher_encoder", False))
        if model.freeze_teacher_encoder:
            for p in model.encoder.parameters():
                p.requires_grad = False
            model.encoder.eval()
    elif args.model_family == "wearecg_vae":
        from src.reconstruction.unified_latents.engineering.vae_fm import WearECGVAE

        model = WearECGVAE(
            latent_channels=payload.get("latent_channels", args.latent_channels),
            target_len=payload.get("target_len", args.target_len),
            beta_kl=payload.get("beta_kl", 1e-4),
            missing_lead_weight=payload.get("missing_lead_weight", args.missing_lead_weight),
        )
        if "encoder_state_dict" in payload and "decoder_state_dict" in payload:
            model.encoder.load_state_dict(payload["encoder_state_dict"], strict=True)
            model.decoder.load_state_dict(payload["decoder_state_dict"], strict=True)
            return model.to(device, dtype=torch.float32)
    else:
        from src.reconstruction.unified_latents.engineering.vae_fm import WearECGFMVAE

        model = WearECGFMVAE(
            fm_checkpoint_path=payload.get("fm_checkpoint", args.fm_checkpoint),
            latent_channels=payload.get("latent_channels", 4),
            target_len=payload.get("target_len", args.target_len),
            beta_kl=payload.get("beta_kl", 1e-4),
            missing_lead_weight=payload.get("missing_lead_weight", args.missing_lead_weight),
            fm_loss_weight=payload.get("fm_loss_weight", 1e-2),
            fm_cosine_mix=payload.get("fm_cosine_mix", 0.5),
            use_decoder_conditioning=payload.get("use_decoder_conditioning", False),
            fm_cond_drop_prob=payload.get("fm_cond_drop_prob", 0.0),
            use_latent_alignment=payload.get("use_latent_alignment", False),
            latent_align_weight=payload.get("latent_align_weight", 1e-3),
        )
    _load_checkpoint_state_strict(model, state_dict, args.model_family)
    return model.to(device, dtype=torch.float32)


def main():
    args = get_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    obs_indices = resolve_obs_leads(args.regime, args.obs_leads)
    payload = _checkpoint_payload(args.checkpoint)
    base_dir = "data/ptb_xl/tensors"
    dataset = TensorFolderDataset(f"{base_dir}/{args.split}")
    if args.debug:
        dataset = torch.utils.data.Subset(dataset, range(64))
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True)
    model = load_model(args, device)
    metrics = evaluate_reconstruction(model, loader, device, obs_indices, split=args.split, model_family=args.model_family)

    output_dir = args.output_dir or os.path.join(
        os.path.dirname(args.checkpoint),
        f"eval_{args.model_family}_{args.split}_{format_lead_set(obs_indices)}",
    )
    metadata = build_eval_metadata(args, obs_indices, payload)
    if args.model_family == "fm_vae":
        metadata.update(
            {
                "fm_checkpoint": payload.get("fm_checkpoint", args.fm_checkpoint),
                "fm_perceptual": payload.get("fm_perceptual", payload.get("fm_loss_weight", 0.0) > 0.0),
                "fm_loss_weight": payload.get("fm_loss_weight", 1e-2),
                "fm_cosine_mix": payload.get("fm_cosine_mix", 0.5),
                "fm_decoder_conditioning": payload.get("use_decoder_conditioning", False),
                "fm_latent_align": payload.get("use_latent_alignment", False),
                "missing_lead_weight": payload.get("missing_lead_weight", args.missing_lead_weight),
                "checkpoint_contains_fm_backbone": False,
            }
        )
    if args.include_sunnybrook or args.include_ecgfounder:
        print("External panels were requested but remain gated until an internal chest-metric win is promoted.")
    write_run_artifacts(output_dir, metadata, metrics)
    write_json(os.path.join(output_dir, "eval_metadata.json"), metadata)
    print(f"Saved evaluation to {output_dir}")
    _print_eval_summary(
        metrics,
        split=args.split,
        model_family=args.model_family,
        obs_indices=obs_indices,
        debug=bool(args.debug),
    )
    for key in [
        f"{args.split}/r2_reg_v4_v6_mean",
        f"{args.split}/r2_reg_v3_v6_mean",
        f"{args.split}/r2_regressor",
        f"{args.split}/r2_teacher_clean",
    ]:
        if key in metrics:
            print(f"{key}: {metrics[key]:.4f}")


if __name__ == "__main__":
    main()


### Sunnybrook Extractor (extract_sunnybrook_features.py)


In [ ]:
"""Sunnybrook ECG metadata extraction pipeline.

This module parses Philips ECG XML files, enforces guardrails around data
integrity and privacy, and exports sanitized feature vectors for downstream
conditioning in the reconstruction model.

Guardrails referenced:
- G1.1: XML validation and error logging
- G1.2: Removal of protected health information
- G1.3: Explicit missingness indicators (no silent imputation)
- G1.4: Physiologic range checks (flag outliers, keep values)
- G1.5: Final CSV must be readable with strict schema (no NaNs written)
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import logging
from logging.handlers import RotatingFileHandler
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple, Union
import xml.etree.ElementTree as ET

import pandas as pd

try:
    import yaml
except ImportError:  # pragma: no cover - yaml is optional
    yaml = None


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PHI_FIELDS = {
    "patient_name",
    "name",
    "mrn",
    "medical_record_number",
    "provider_name",
    "email",
    "phone",
    "dob",
}

DEFAULT_OUTLIER_THRESHOLDS: Dict[str, Tuple[Optional[float], Optional[float]]] = {
    "age_years": (0, 120),
    "qrs_duration_ms": (50, 300),
    "qt_interval_ms": (200, 600),
    "pr_interval_ms": (80, 400),
    "qrs_axis_deg": (-180, 180),
    "t_wave_axis_deg": (-180, 180),
    "sampling_rate_hz": (100, 2000),
    "heart_rate_bpm": (20, 200),
}

DEFAULT_XPATH_MAP: Dict[str, List[str]] = {
    # Demographics
    "patient_id": [
        ".//{ns}PatientID",
        ".//{ns}patientId",
        ".//patient/id",
        ".//Case/CaseId",
    ],
    "encounter_id": [
        ".//{ns}VisitID",
        ".//visitId",
        ".//Case/EncounterId",
    ],
    "age_years": [
        ".//{ns}PatientAge",
        ".//patientAge",
        ".//Patient/Age",
    ],
    "sex_code": [
        ".//{ns}PatientSex",
        ".//patientSex",
        ".//Patient/Sex",
    ],
    "pacemaker_status": [
        ".//{ns}Pacemaker",
        ".//pacemaker",
        ".//Findings/Pacemaker",
    ],
    # Acquisition
    "sampling_rate_hz": [
        ".//{ns}SampleBase",
        ".//SamplingRate",
        ".//acquisition/SamplingRate",
    ],
    "lowpass_hz": [
        ".//{ns}LowPassFilter",
        ".//Filters/LowPass",
    ],
    "hipass_hz": [
        ".//{ns}HighPassFilter",
        ".//Filters/HighPass",
    ],
    "machine_model": [
        ".//{ns}MachineModel",
        ".//Device/Model",
    ],
    # Intervals
    "qrs_duration_ms": [
        ".//{ns}QRSDuration",
        ".//Intervals/QRS",
    ],
    "qt_interval_ms": [
        ".//{ns}QTInterval",
        ".//Intervals/QT",
    ],
    "pr_interval_ms": [
        ".//{ns}PRInterval",
        ".//Intervals/PR",
    ],
    # Axes
    "qrs_axis_deg": [
        ".//{ns}QRSAxis",
        ".//Axis/QRS",
    ],
    "t_wave_axis_deg": [
        ".//{ns}TAxis",
        ".//Axis/T",
    ],
}

DEFAULT_NULL_TOKEN = "NULL"


@dataclass
class SunnybrookExtractorConfig:
    """Configuration container for the Sunnybrook feature extractor."""

    input_dir: Path
    features_csv: Path
    raw_csv: Path
    error_log: Path
    warning_csv: Path
    qa_report_path: Path
    qa_figure_path: Path
    hash_salt: str = "sunnybrook-default-salt"
    null_token: str = DEFAULT_NULL_TOKEN
    outlier_thresholds: Dict[str, Tuple[Optional[float], Optional[float]]] = field(
        default_factory=lambda: DEFAULT_OUTLIER_THRESHOLDS.copy()
    )
    xpath_map: Dict[str, List[str]] = field(
        default_factory=lambda: {k: v[:] for k, v in DEFAULT_XPATH_MAP.items()}
    )
    ensure_directories: bool = True

    @classmethod
    def from_mapping(cls, data: Dict[str, Union[str, Dict, List]]) -> "SunnybrookExtractorConfig":
        """Build config from dictionary, coercing paths to Path objects."""

        def to_path(value: Union[str, Path]) -> Path:
            return Path(value).expanduser().resolve()

        required_keys = [
            "input_dir",
            "features_csv",
            "raw_csv",
            "error_log",
            "warning_csv",
            "qa_report_path",
            "qa_figure_path",
        ]

        missing = [k for k in required_keys if k not in data]
        if missing:
            raise ValueError(f"Missing required config keys: {missing}")

        mapped = {
            "input_dir": to_path(data["input_dir"]),
            "features_csv": to_path(data["features_csv"]),
            "raw_csv": to_path(data["raw_csv"]),
            "error_log": to_path(data["error_log"]),
            "warning_csv": to_path(data["warning_csv"]),
            "qa_report_path": to_path(data["qa_report_path"]),
            "qa_figure_path": to_path(data["qa_figure_path"]),
        }
        optional_keys = {"hash_salt", "null_token", "outlier_thresholds", "xpath_map", "ensure_directories"}
        for key in optional_keys:
            if key in data:
                mapped[key] = data[key]

        return cls(**mapped)

    @classmethod
    def from_file(cls, path: Union[str, Path]) -> "SunnybrookExtractorConfig":
        """Load configuration from JSON or YAML file."""
        path = Path(path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"Config file not found: {path}")

        if path.suffix.lower() in {".yaml", ".yml"}:
            if yaml is None:
                raise RuntimeError("pyyaml not available but YAML config requested")
            with path.open("r", encoding="utf-8") as fh:
                data = yaml.safe_load(fh)
        else:
            with path.open("r", encoding="utf-8") as fh:
                data = json.load(fh)

        if not isinstance(data, dict):
            raise ValueError(f"Config file must contain mapping, got {type(data)}")

        return cls.from_mapping(data)


# ---------------------------------------------------------------------------
# Utility functions
# ---------------------------------------------------------------------------

def _build_logger(name: str, error_log: Path) -> logging.Logger:
    """Create a logger that writes INFO to stdout and errors to file."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)

    if logger.handlers:
        return logger  # reuse existing logger

    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    formatter = logging.Formatter("[%(asctime)s] %(levelname)s %(name)s: %(message)s")
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    error_log.parent.mkdir(parents=True, exist_ok=True)
    file_handler = RotatingFileHandler(error_log, maxBytes=5 * 1024 * 1024, backupCount=3)
    file_handler.setLevel(logging.ERROR)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    logger.propagate = False
    return logger


def _hash_value(raw: str, salt: str) -> str:
    """Return SHA256 hash of the supplied value using provided salt."""
    hasher = hashlib.sha256()
    hasher.update(salt.encode("utf-8"))
    hasher.update(str(raw).encode("utf-8"))
    return hasher.hexdigest()


def _md5_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Compute md5 hash of a file (used for provenance)."""
    digest = hashlib.md5()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _ensure_warning_csv(path: Path) -> None:
    """Ensure warning CSV exists with header."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="", encoding="utf-8") as fh:
            writer = csv.writer(fh)
            writer.writerow(["timestamp_utc", "file", "field", "value", "expected_range", "note"])


# ---------------------------------------------------------------------------
# Extractor implementation
# ---------------------------------------------------------------------------

class SunnybrookFeatureExtractor:
    """Parser and processor for Sunnybrook Philips ECG XML metadata."""

    def __init__(self, config: SunnybrookExtractorConfig) -> None:
        self.config = config
        self.logger = _build_logger(self.__class__.__name__, config.error_log)
        self.invalid_files: List[Path] = []
        self.stats: Dict[str, int] = {
            "total_files": 0,
            "parsed": 0,
            "skipped": 0,
            "outliers": 0,
        }
        self.xpath_map = config.xpath_map
        self.null_token = config.null_token
        self.hash_salt = config.hash_salt
        _ensure_warning_csv(config.warning_csv)
        if config.ensure_directories:
            config.features_csv.parent.mkdir(parents=True, exist_ok=True)
            config.raw_csv.parent.mkdir(parents=True, exist_ok=True)
            config.qa_report_path.parent.mkdir(parents=True, exist_ok=True)
            config.qa_figure_path.parent.mkdir(parents=True, exist_ok=True)

    # ---------------------------- XML helpers ----------------------------

    def validate_schema(self, xml_path: Path) -> Tuple[bool, Optional[ET.ElementTree], Dict[str, str]]:
        """Validate XML structure (G1.1). Returns (is_valid, tree, namespaces)."""
        try:
            tree = ET.parse(xml_path)
        except (ET.ParseError, OSError) as exc:
            self.logger.error("Failed to parse %s: %s", xml_path, exc)
            self.invalid_files.append(xml_path)
            self.stats["skipped"] += 1
            return False, None, {}

        root = tree.getroot()
        ns_map = self._build_namespace_map(root)
        return True, tree, ns_map

    @staticmethod
    def _build_namespace_map(root: ET.Element) -> Dict[str, str]:
        """Extract namespace mappings from the XML root element."""
        ns_map: Dict[str, str] = {}
        if root.tag.startswith("{"):
            uri = root.tag.split("}")[0][1:]
            ns_map["ns"] = uri
        # Additional namespace definitions on elements
        for key, value in root.attrib.items():
            if key.startswith("xmlns:"):
                prefix = key.split("xmlns:")[1]
                ns_map[prefix] = value
            elif key == "xmlns":
                ns_map["ns"] = value
        if "ns" not in ns_map:
            ns_map["ns"] = ""  # fallback for non-namespaced docs
        return ns_map

    def _find_text(self, root: ET.Element, ns_map: Dict[str, str], candidates: Iterable[str]) -> Optional[str]:
        """Attempt multiple xpaths (with namespace substitution) to retrieve text."""
        for raw_xpath in candidates:
            xpath = (
                raw_xpath.format(ns="{%s}" % ns_map["ns"])
                if "{ns}" in raw_xpath and ns_map.get("ns")
                else raw_xpath.replace("{ns}", "")
            )
            element = root.find(xpath, namespaces=ns_map if ns_map.get("ns") else None)
            if element is None:
                continue
            text = element.text
            if text is not None:
                stripped = text.strip()
                if stripped:
                    return stripped
        return None

    def parse_single_ecg(
        self, tree: ET.ElementTree, xml_path: Path, ns_map: Dict[str, str]
    ) -> Optional[Dict[str, Union[str, float, int]]]:
        """Extract metadata from a single ECG XML file (G1.2-G1.4)."""
        root = tree.getroot()
        record: Dict[str, Union[str, float, int]] = {
            "file": str(xml_path.resolve()),
            "processing_timestamp": datetime.now(timezone.utc).isoformat(),
            "source_md5": _md5_file(xml_path),
        }

        missing_flags: Dict[str, int] = {}

        # Extract fields defined in xpath_map
        for field, candidates in self.xpath_map.items():
            value = self._find_text(root, ns_map, candidates)
            if value is None:
                record[field] = self.null_token
                missing_flags[f"is_missing_{field}"] = 1
                continue
            cleaned = value.strip()
            record[field] = cleaned
            missing_flags[f"is_missing_{field}"] = 0

        # Hash identifiers (prevent PHI leakage)
        for key in ["patient_id", "encounter_id", "machine_model"]:
            if key in record:
                value = record[key]
                if value == self.null_token:
                    continue
                hashed = _hash_value(value, self.hash_salt)
                record[f"{key}_hash"] = hashed
                del record[key]
                missing_flags.setdefault(f"is_missing_{key}", 0)

        # Ensure encounter hash fallback
        if "encounter_id_hash" not in record:
            fallback = _hash_value(xml_path.stem, self.hash_salt)
            record["encounter_id_hash"] = fallback
            missing_flags.setdefault("is_missing_encounter_id", 1)

        # Pacemaker categorical normalization (0=unknown,1=no,2=yes)
        record["pacemaker_status"] = self._normalize_categorical(record.get("pacemaker_status", self.null_token))
        missing_flags.setdefault("is_missing_pacemaker_status", 0 if record["pacemaker_status"] != 0 else 1)

        # Sex code normalization
        record["sex_code"] = self._normalize_sex(record.get("sex_code", self.null_token))
        missing_flags.setdefault("is_missing_sex_code", 0 if record["sex_code"] != 0 else 1)

        # Numeric conversions + outlier logging (G1.4)
        numeric_fields = [
            "age_years",
            "sampling_rate_hz",
            "lowpass_hz",
            "hipass_hz",
            "qrs_duration_ms",
            "qt_interval_ms",
            "pr_interval_ms",
            "qrs_axis_deg",
            "t_wave_axis_deg",
            "heart_rate_bpm",
        ]
        for field in numeric_fields:
            raw_value = record.get(field, self.null_token)
            if raw_value == self.null_token:
                continue
            converted = self._safe_float(raw_value)
            if converted is None:
                self.logger.warning("Non-numeric value for %s in %s: %s", field, xml_path, raw_value)
                record[field] = self.null_token
                missing_flags[f"is_missing_{field}"] = 1
                continue
            record[field] = converted
            bounds = self.config.outlier_thresholds.get(field)
            if bounds is not None:
                low, high = bounds
                out_of_range = (low is not None and converted < low) or (high is not None and converted > high)
                if out_of_range:
                    self._log_outlier(xml_path, field, converted, bounds, note="Physiologic range violation")
                    self.stats["outliers"] += 1

        # Check PHI
        self._assert_no_phi(record)

        # Merge missing indicators
        record.update(missing_flags)

        namespace = ns_map.get("ns", "")
        prefix = f"{{{namespace}}}" if namespace else ""

        # Enrich demographics
        patient_info = root.find(f".//{prefix}patient/{prefix}generalpatientdata")
        if patient_info is not None:
            years_el = patient_info.find(f".//{prefix}age/{prefix}years")
            if years_el is not None and years_el.text:
                try:
                    record["age_years"] = float(years_el.text)
                    missing_flags[f"is_missing_age_years"] = 0
                except ValueError:
                    pass
            sex_el = patient_info.find(f".//{prefix}sex")
            if sex_el is not None and sex_el.text:
                sex_text = sex_el.text.strip().lower()
                if "male" in sex_text:
                    record["sex_code"] = 1
                elif "female" in sex_text:
                    record["sex_code"] = 2
                elif "unknown" in sex_text:
                    record["sex_code"] = 0
                missing_flags["is_missing_sex_code"] = 0

        # Activity measurements
        measurements = root.find(f".//{prefix}internalmeasurements/{prefix}crossleadmeasurements")
        measurement_map = {
            "meanventrate": "heart_rate_bpm",
            "meanqrsdur": "qrs_duration_ms",
            "meanqtint": "qt_interval_ms",
            "meanprint": "pr_interval_ms",
            "qrsfrontaxis": "qrs_axis_deg",
            "tfrontaxis": "t_wave_axis_deg",
        }
        if measurements is not None:
            for xml_tag, key in measurement_map.items():
                el = measurements.find(f"{prefix}{xml_tag}")
                if el is not None and el.text:
                    try:
                        record[key] = float(el.text)
                        missing_flags[f"is_missing_{key}"] = 0
                    except ValueError:
                        continue
        if "heart_rate_bpm" not in record:
            record["heart_rate_bpm"] = self.null_token
        missing_flags.setdefault("is_missing_heart_rate_bpm", 1 if record["heart_rate_bpm"] == self.null_token else 0)

        # Acquisition metadata
        wave = root.find(f".//{prefix}waveforms/{prefix}parsedwaveforms")
        if wave is not None:
            attr_map = {
                "samplespersecond": "sampling_rate_hz",
                "hipass": "hipass_hz",
                "lowpass": "lowpass_hz",
            }
            for attr, key in attr_map.items():
                value = wave.get(attr)
                if value not in (None, "", self.null_token):
                    try:
                        record[key] = float(value)
                        missing_flags[f"is_missing_{key}"] = 0
                    except ValueError:
                        continue

        return record

    @staticmethod
    def _safe_float(value: Union[str, float, int]) -> Optional[float]:
        """Convert value to float, returning None on failure."""
        try:
            return float(str(value).strip())
        except (TypeError, ValueError):
            return None

    def _normalize_categorical(self, raw: Union[str, int]) -> int:
        """Normalize pacemaker categorical values."""
        if isinstance(raw, int):
            if raw in {0, 1, 2}:
                return raw
            return 0
        if not isinstance(raw, str):
            return 0
        lowered = raw.strip().lower()
        if lowered in {"1", "no", "false", "absent"}:
            return 1
        if lowered in {"2", "yes", "true", "present"}:
            return 2
        if lowered in {"0", "unknown", "", "n/a"}:
            return 0
        return 0

    def _normalize_sex(self, raw: Union[str, int]) -> int:
        """Normalize sex categorical values (0=unknown,1=male,2=female)."""
        if isinstance(raw, int):
            if raw in {0, 1, 2}:
                return raw
            return 0
        lowered = str(raw).strip().lower()
        if lowered in {"m", "male", "1"}:
            return 1
        if lowered in {"f", "female", "2"}:
            return 2
        return 0

    def _log_outlier(
        self,
        xml_path: Path,
        field: str,
        value: Union[int, float],
        bounds: Tuple[Optional[float], Optional[float]],
        note: str,
    ) -> None:
        """Append outlier information to CSV (G1.4)."""
        with self.config.warning_csv.open("a", newline="", encoding="utf-8") as fh:
            writer = csv.writer(fh)
            writer.writerow(
                [
                    datetime.now(timezone.utc).isoformat(),
                    str(xml_path.resolve()),
                    field,
                    value,
                    str(bounds),
                    note,
                ]
            )

    def _assert_no_phi(self, record: Dict[str, Union[str, float, int]]) -> None:
        """Ensure no PHI-like keys are present (G1.2)."""
        present = PHI_FIELDS.intersection(record.keys())
        if present:
            raise ValueError(f"PHI fields detected in record: {present}")

    # ---------------------------- orchestration ----------------------------

    def process_all(self, sample_n: Optional[int] = None) -> pd.DataFrame:
        """Process all XML files and return sanitized DataFrame."""
        xml_files = sorted(Path(self.config.input_dir).glob("*.xml"))
        if not xml_files:
            self.logger.warning("No XML files found under %s", self.config.input_dir)
            empty_df = pd.DataFrame()
            return empty_df, empty_df

        records: List[Dict[str, Union[str, float, int]]] = []
        for idx, xml_path in enumerate(xml_files, start=1):
            if sample_n is not None and idx > sample_n:
                break
            self.stats["total_files"] += 1
            if idx % 25 == 0:
                self.logger.info("Processing file %d of %d", idx, len(xml_files))
            valid, tree, ns_map = self.validate_schema(xml_path)
            if not valid or tree is None:
                continue
            record = self.parse_single_ecg(tree, xml_path, ns_map)
            if record is None:
                self.stats["skipped"] += 1
                continue
            records.append(record)
            self.stats["parsed"] += 1

        raw_df, export_df = self._build_dataframe(records)
        self._write_outputs(raw_df, export_df)
        return raw_df, export_df

    def _build_dataframe(self, records: List[Dict[str, Union[str, float, int]]]) -> pd.DataFrame:
        """Build DataFrame with consistent column ordering."""
        if not records:
            empty_df = pd.DataFrame()
            return empty_df, empty_df

        df = pd.DataFrame(records)
        df["processing_timestamp"] = pd.to_datetime(df["processing_timestamp"], utc=True)

        # Ensure missing indicator columns exist
        missing_cols = [col for col in df.columns if col.startswith("is_missing_")]
        for field in self.xpath_map.keys():
            flag_col = f"is_missing_{field}"
            if flag_col not in missing_cols:
                df[flag_col] = 1

        # Convert numeric columns to floats, others to strings
        numeric_cols = [
            "age_years",
            "sampling_rate_hz",
            "lowpass_hz",
            "hipass_hz",
            "qrs_duration_ms",
            "qt_interval_ms",
            "pr_interval_ms",
            "qrs_axis_deg",
            "t_wave_axis_deg",
            "heart_rate_bpm",
        ]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # Replace NaN with null token for export
        df_export = df.copy()
        for col in numeric_cols:
            if col in df_export.columns:
                df_export[col] = df_export[col].apply(
                    lambda x: self.null_token if pd.isna(x) else f"{x:.6g}"
                )

        additional_columns = [
            "file",
            "encounter_id_hash",
            "pacemaker_status",
            "sex_code",
            "processing_timestamp",
            "source_md5",
        ]
        for col in additional_columns:
            if col not in df_export.columns:
                df_export[col] = self.null_token

        # Append hashed machine model if present
        if "machine_model_hash" not in df_export.columns:
            df_export["machine_model_hash"] = self.null_token

        # Ensure boolean indicators exported as 0/1 strings
        indicator_cols = [c for c in df_export.columns if c.startswith("is_missing_")]
        for col in indicator_cols:
            df_export[col] = df_export[col].fillna(1).astype(int).astype(str)

        # Convert processing timestamp to ISO string for export
        df_export["processing_timestamp"] = df["processing_timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")

        # Ensure column order: base identifiers, numeric, categorical, indicators
        identifier_cols = [
            "file",
            "encounter_id_hash",
            "machine_model_hash",
            "processing_timestamp",
            "source_md5",
        ]
        categorical_cols = ["sex_code", "pacemaker_status"]
        column_order = (
            identifier_cols
            + [col for col in numeric_cols if col in df_export.columns]
            + categorical_cols
            + sorted(indicator_cols)
        )
        # Add any remaining columns (e.g., fallback hashes)
        remaining = [col for col in df_export.columns if col not in column_order]
        column_order.extend(sorted(remaining))
        df_export = df_export[column_order]
        return df, df_export

    def _write_outputs(self, raw_df: pd.DataFrame, export_df: pd.DataFrame) -> None:
        """Persist raw and sanitized DataFrames (G1.5 validation)."""
        if raw_df.empty:
            self.logger.warning("No records parsed; skipping CSV export.")
            return

        export_df = export_df.fillna(self.null_token)

        raw_df.to_csv(self.config.raw_csv, index=False)
        export_df.to_csv(self.config.features_csv, index=False)

        # Validate exported CSV by reading back in
        try:
            validation_df = pd.read_csv(
                self.config.features_csv,
                dtype=str,
                keep_default_na=False,
                na_values=[],
            )
        except Exception as exc:  # pragma: no cover - defensive
            self.logger.error("Failed to read back exported CSV (G1.5 violation): %s", exc)
            raise

        expected_cols = set(export_df.columns)
        if set(validation_df.columns) != expected_cols:
            raise ValueError(
                f"CSV validation failed: columns mismatch. "
                f"expected={sorted(expected_cols)}, found={sorted(validation_df.columns)}"
            )

        if validation_df.isnull().any().any():
            null_cols = validation_df.columns[validation_df.isnull().any()].tolist()
            raise ValueError(
                f"CSV validation failed: detected NaN values after export (G1.5) in columns {null_cols}"
            )

        self.logger.info(
            "Exported raw features to %s and sanitized features to %s",
            self.config.raw_csv,
            self.config.features_csv,
        )

        self._generate_quality_artifacts(raw_df, export_df)

    # ---------------------------- QA / Reporting ----------------------------

    def _generate_quality_artifacts(self, raw_df: pd.DataFrame, export_df: pd.DataFrame) -> None:
        """Generate QA report and distribution plots (G1.3, G1.4)."""
        import matplotlib.pyplot as plt  # Local import to avoid heavy dependency at import time

        report_lines: List[str] = []
        report_lines.append(f"Sunnybrook Metadata QA Report - {datetime.now(timezone.utc).isoformat()}")
        report_lines.append("=" * 72)
        report_lines.append(f"Total samples processed: {len(export_df)}")
        report_lines.append(f"Files skipped (invalid XML): {self.stats['skipped']}")
        report_lines.append(f"Outliers flagged: {self.stats['outliers']}")
        report_lines.append("")

        # Demographics summary
        def summarize(series: pd.Series) -> str:
            if series.dropna().empty:
                return "no data"
            return (
                f"mean={series.mean():.2f}, std={series.std():.2f}, "
                f"min={series.min():.2f}, max={series.max():.2f}, n={series.count()}"
            )

        numeric_cols = [
            "age_years",
            "qrs_duration_ms",
            "qt_interval_ms",
            "pr_interval_ms",
            "heart_rate_bpm",
        ]
        for col in numeric_cols:
            if col in raw_df.columns:
                series = pd.to_numeric(raw_df[col], errors="coerce")
                report_lines.append(f"{col}: {summarize(series)}")
        report_lines.append("")

        # Missingness stats
        indicator_cols = [c for c in export_df.columns if c.startswith("is_missing_")]
        report_lines.append("Missingness (% of records):")
        for col in sorted(indicator_cols):
            percent_missing = export_df[col].astype(float).mean() * 100.0
            report_lines.append(f"  {col}: {percent_missing:.2f}%")
        report_lines.append("")

        # Guardrail compliance summary
        report_lines.append("Guardrail Compliance:")
        report_lines.append("  G1.1 XML validation executed (see logs/parsing_errors.log)")
        report_lines.append("  G1.2 PHI scrubbing enforced via hashed identifiers")
        report_lines.append("  G1.3 Missingness indicators exported")
        report_lines.append("  G1.4 Physiology checks logged to warnings/outliers.csv")
        report_lines.append("  G1.5 CSV round-trip validation passed")
        report_lines.append("")

        self.config.qa_report_path.write_text("\n".join(report_lines), encoding="utf-8")

        # Distribution plots
        plt.style.use("ggplot")
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        plot_fields = ["age_years", "qrs_duration_ms", "qt_interval_ms"]
        thresholds = self.config.outlier_thresholds
        for ax, field in zip(axes, plot_fields):
            series = pd.to_numeric(raw_df.get(field, pd.Series(dtype=float)), errors="coerce").dropna()
            if series.empty:
                ax.text(0.5, 0.5, f"No data for {field}", ha="center", va="center")
                continue
            ax.hist(series, bins=20, color="#2E86AB", alpha=0.8)
            ax.set_title(field)
            ax.set_xlabel(field)
            ax.set_ylabel("Count")
            bounds = thresholds.get(field)
            if bounds:
                low, high = bounds
                if low is not None:
                    ax.axvline(low, color="red", linestyle="--", label="min threshold")
                if high is not None:
                    ax.axvline(high, color="red", linestyle="--", label="max threshold")
            ax.axvline(series.mean(), color="black", linestyle="-", linewidth=1, label="mean")
            ax.legend(loc="upper right")

        fig.tight_layout()
        fig.savefig(self.config.qa_figure_path, dpi=200)
        plt.close(fig)

        self.logger.info("Wrote QA report to %s and distributions to %s", self.config.qa_report_path, self.config.qa_figure_path)

    # ------------------------------------------------------------------
    # CLI
    # ------------------------------------------------------------------

    @staticmethod
    def add_cli(parser: argparse.ArgumentParser) -> None:
        """Register CLI options."""
        parser.add_argument("--config", required=True, help="Path to extractor config (JSON or YAML).")
        parser.add_argument("--sample_n", type=int, default=None, help="Optional cap on number of XML files for smoke test.")

    @classmethod
    def from_cli(cls) -> "SunnybrookFeatureExtractor":
        """Instantiate extractor using CLI arguments."""
        parser = argparse.ArgumentParser(description="Sunnybrook ECG metadata extractor")
        cls.add_cli(parser)
        args = parser.parse_args()
        config = SunnybrookExtractorConfig.from_file(args.config)
        extractor = cls(config)
        extractor._cli_sample_n = args.sample_n  # attach for run()
        return extractor

    def run(self, sample_n: Optional[int] = None) -> None:
        """Execute extraction pipeline and update PHASE_STATUS log."""
        if sample_n is None:
            sample_n = getattr(self, "_cli_sample_n", None)

        raw_df, export_df = self.process_all(sample_n=sample_n)
        if export_df.empty:
            self.logger.warning("No data exported; skipping PHASE_STATUS update.")
            return

        # Update phase status file if available
        status_path = Path("PHASE_STATUS.txt")
        if status_path.exists():
            self._update_phase_status(status_path, len(export_df))

    def _update_phase_status(self, status_path: Path, num_records: int) -> None:
        """Append Phase 1 completion summary to PHASE_STATUS.txt."""
        timestamp = datetime.now(timezone.utc).isoformat()
        summary_lines = [
            "",
            f"[{timestamp}] Phase 1 Extraction Summary",
            f"Processed files: {self.stats['total_files']}",
            f"Parsed records: {self.stats['parsed']}",
            f"Skipped files: {self.stats['skipped']}",
            f"Outliers flagged: {self.stats['outliers']}",
            f"Exported records: {num_records}",
            f"Features CSV: {self.config.features_csv}",
            f"Raw CSV: {self.config.raw_csv}",
            f"Error log: {self.config.error_log}",
            f"Warnings CSV: {self.config.warning_csv}",
            "Guardrail checks:",
            "  [x] G1.1",
            "  [x] G1.2",
            "  [x] G1.3",
            "  [x] G1.4",
            "  [x] G1.5",
        ]
        with status_path.open("a", encoding="utf-8") as fh:
            fh.write("\n".join(summary_lines) + "\n")


def main() -> None:
    extractor = SunnybrookFeatureExtractor.from_cli()
    extractor.run()


if __name__ == "__main__":  # pragma: no cover
    main()

